# 🚀 AI-SDLC Accelerator — Complete Code Walkthrough

**Purpose:** This notebook is a step-by-step teaching guide for the entire AI-SDLC Accelerator project.  
You can run each cell sequentially to understand how every component works.

## What This Project Does
A **multi-agent AI system** that automates the Software Development Lifecycle:
1. **BA Agent** — Chats with you to gather API requirements
2. **Scrum Master Agent** — Creates planning docs (Jira, Confluence, Flowchart, README) + orchestrates all agents
3. **Developer Agent** — Generates a full FastAPI project
4. **Tester Agent** — Generates and runs self-contained pytest tests
5. **DevOps Agent** — Generates Docker/K8s/CI-CD configs and creates a GitHub PR

## Architecture
```
User → BA Agent (chat) → Structured Spec → LangGraph Pipeline
                                                │
                                    ┌───────────┼───────────┐
                                    ▼           ▼           ▼
                                Developer   Tester      DevOps
                                    │           │           │
                                    └─────── Orchestrator ──┘
                                          (Scrum Master)
```

## How to Use This Notebook
- Run cells **in order** from top to bottom
- Each section has detailed comments explaining the code
- Some cells define classes/functions; the execution cells come later
- Make sure your `.env` file is configured before running

---
# 📦 SECTION 1: Setup & Imports
---
Before we begin, we need to:
1. Set our working directory to the project root (so all relative imports work)
2. Import all libraries we'll use throughout the notebook

In [238]:
# ============================================================================
# STEP 1.1: Set working directory to project root
# ============================================================================
#
# WHY THIS CELL EXISTS:
# --------------------
# This notebook lives inside the /notebooks/ folder, but all our source code
# (config/, agents/, rag/, utils/, etc.) lives in the project root folder.
# Python resolves imports relative to the CURRENT WORKING DIRECTORY (CWD).
# If the CWD is /notebooks/, then "from config.settings import get_settings"
# would FAIL because Python would look for /notebooks/config/settings.py
# which doesn't exist.
#
# WHAT WE DO:
# ----------
# 1. Detect the project root directory (one level up from /notebooks/)
# 2. Change the CWD to the project root using os.chdir()
# 3. Add the project root to sys.path so Python can find our modules
#
# HOW WE DETECT THE PROJECT ROOT:
# VS Code injects a special variable __vsc_ipynb_file__ that contains
# the full path to this notebook file. We use that to reliably navigate
# to the project root, regardless of where the kernel started.

import os
import sys

# Get the notebook's actual file path from the Jupyter/IPython kernel.
# __vsc_ipynb_file__ is a VS Code-specific variable — it's the most
# reliable way to know WHERE this notebook file lives on disk.
try:
    _nb_path = __vsc_ipynb_file__          # VS Code injects this variable
except NameError:
    _nb_path = None  # Not running in VS Code — use fallback below

if _nb_path:
    # _nb_path = "d:\...\Project 13 - AI in SDLC\notebooks\AI_SDLC_Walkthrough.ipynb"
    # dirname once → "d:\...\Project 13 - AI in SDLC\notebooks\"
    # dirname twice → "d:\...\Project 13 - AI in SDLC\"  ← this is PROJECT_ROOT
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(_nb_path)))
else:
    # Fallback: hardcoded absolute path (edit this if your project path differs)
    PROJECT_ROOT = r"d:\Users data\Documents\AI Engnieeering Projects\Project 13 - AI in SDLC"

# Change the current working directory to project root
# After this, all relative paths like "guideline.txt" and "guidelines/" resolve correctly
os.chdir(PROJECT_ROOT)

# Add project root to Python path so "from config.settings import ..." works
# sys.path is the list of directories Python searches when you do an import
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Verify everything is set up correctly
print(f"✅ Working directory set to: {os.getcwd()}")
print(f"✅ Python path includes: {PROJECT_ROOT}")
print(f"✅ guideline.txt exists: {os.path.isfile('guideline.txt')}")
print(f"✅ guidelines/ folder exists: {os.path.isdir('guidelines')}")

✅ Working directory set to: d:\Users data\Documents\AI Engnieeering Projects\Project 13 - AI in SDLC
✅ Python path includes: d:\Users data\Documents\AI Engnieeering Projects\Project 13 - AI in SDLC
✅ guideline.txt exists: True
✅ guidelines/ folder exists: True


In [239]:
# ============================================================================
# STEP 1.2: Import all libraries
# ============================================================================
#
# WHY ALL IMPORTS ARE HERE (not spread across cells):
# ---------------------------------------------------
# In the actual project, each file imports only what it needs.
# But in a notebook, if you import inside a cell and that cell fails,
# you'd have to re-run it. By putting ALL imports in one cell at the top,
# we ensure everything is available for every subsequent cell.
#
# THE LIBRARY STACK:
# ------------------
# 1. Python stdlib: json, re, sqlite3, subprocess, os, sys
# 2. LangChain: LLM wrapper, message types, document handling, text splitting
# 3. LangGraph: StateGraph for multi-agent pipeline, checkpointing
# 4. FAISS: Facebook's vector similarity search (for RAG)
# 5. CrossEncoder: sentence-transformers model for reranking RAG results
# 6. dotenv: loads .env file into environment variables

import json                              # JSON parsing/serialization — every agent returns JSON
import re                                # Regular expressions — cleaning LLM output, git branch names
import sqlite3                           # SQLite database — our local DB engine (no server needed)
import subprocess                        # Running pytest as a child process (Tester Agent uses this)
import time                              # Timing operations (optional, for benchmarking)

# --- LangChain (the AI framework) ---
# ChatOpenAI: wrapper around OpenAI's chat API — we create one instance PER AGENT
# OpenAIEmbeddings: converts text to 1536-dimensional vectors for FAISS similarity search
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Message types: LangChain uses typed messages to maintain conversation structure
# HumanMessage = user input, SystemMessage = system prompt, AIMessage = LLM response
from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, BaseMessage
)

# JsonOutputParser: LangChain's built-in JSON parser (used as Tier 2 fallback)
from langchain_core.output_parsers import JsonOutputParser

# Document: LangChain's container for text + metadata (used in RAG pipeline)
from langchain_core.documents import Document

# RecursiveCharacterTextSplitter: intelligently splits documents at natural boundaries
# (sections → paragraphs → sentences → words) while respecting chunk_size limits
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TextLoader: reads a .txt file and wraps it as a LangChain Document
from langchain_community.document_loaders import TextLoader

# FAISS: Facebook AI Similarity Search — stores embeddings and finds nearest neighbors
# This is what makes RAG fast — instead of searching all text, we search vectors
from langchain_community.vectorstores import FAISS

# --- LangGraph (the multi-agent orchestration framework) ---
# StateGraph: defines a graph of nodes (agents) and edges (transitions)
# END: a special sentinel node that terminates the pipeline
from langgraph.graph import StateGraph, END

# add_messages: a reducer function that APPENDS new messages to the list
# (instead of overwriting). This is how conversation history accumulates.
from langgraph.graph.message import add_messages

# MemorySaver: in-memory checkpointer — allows the graph to save/restore state
# at each node. Useful for debugging and resuming interrupted pipelines.
from langgraph.checkpoint.memory import MemorySaver

# CrossEncoder: a sentence-transformers model that scores (query, document) pairs
# Used in RAG Stage 2 to RERANK FAISS results for better precision
from sentence_transformers import CrossEncoder

# Type hints for the AgentState TypedDict
from typing import TypedDict, Annotated, Sequence

# dotenv: loads key=value pairs from .env file into os.environ
from dotenv import load_dotenv

print("✅ All imports successful!")

✅ All imports successful!


---
# ⚙️ SECTION 2: Configuration (config/settings.py)
---
The `Settings` class loads all configuration from environment variables (`.env` file).  
This is the **first thing** that gets loaded — every agent reads from this.

**What's in .env:**
- `OPENAI_API_KEY` — Your OpenAI API key (GPT-4o)
- `OPENAI_MODEL` — Which model to use (default: gpt-4o)
- `GITHUB_TOKEN` — For creating Pull Requests
- `GITHUB_REPO_OWNER` / `GITHUB_REPO_NAME` — Target GitHub repository
- `OUTPUT_DIR` — Where generated files are saved (default: "output")
- `GUIDELINES_PATH` — Path to company guidelines document
- `DATABASE_PATH` — Path to SQLite database

In [240]:
# ============================================================================
# STEP 2.1: Settings Class  (config/settings.py)
# ============================================================================
#
# WHAT THIS DOES:
# ---------------
# Reads ALL configuration from environment variables (loaded from .env file).
# Every single component in the project — agents, RAG, database, git — calls
# get_settings() to get its config. This is the SINGLE SOURCE OF TRUTH.
#
# WHY USE ENVIRONMENT VARIABLES?
# ------------------------------
# 1. Security: API keys never appear in source code (only in .env, which is .gitignored)
# 2. Flexibility: Change model, output dir, etc. without editing code
# 3. Best practice: 12-factor app methodology — config in environment, not code
#
# HOW IT FLOWS IN THE PIPELINE:
# ----------------------------
# .env file → load_dotenv() → os.environ → Settings class → get_settings()
#                                                              ↓
#                                              Every agent reads from here

load_dotenv()  # Load .env file → populates os.environ with key=value pairs


class Settings:
    """
    Application settings loaded from environment variables.
    
    Each attribute reads from os.getenv("KEY", "default_value").
    If the env var isn't set, the default is used.
    """
    
    # --- OpenAI Configuration ---
    # OPENAI_API_KEY: authenticates with OpenAI's API for GPT-4o calls
    # Every agent creates its own ChatOpenAI(api_key=settings.OPENAI_API_KEY)
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    
    # OPENAI_MODEL: which model to use — gpt-4o is the default
    # gpt-4o is best for code generation; you could also use gpt-4o-mini for speed
    OPENAI_MODEL: str = os.getenv("OPENAI_MODEL", "gpt-4o")
    
    # --- GitHub Configuration (used by DevOps Agent for PR creation) ---
    # These are optional — if not set, the DevOps Agent skips the PR step
    GITHUB_TOKEN: str = os.getenv("GITHUB_TOKEN", "")
    GITHUB_REPO_OWNER: str = os.getenv("GITHUB_REPO_OWNER", "")
    GITHUB_REPO_NAME: str = os.getenv("GITHUB_REPO_NAME", "")
    GITHUB_BASE_BRANCH: str = os.getenv("GITHUB_BASE_BRANCH", "main")
    
    # --- LangSmith Tracing (optional — for debugging LangChain calls) ---
    # If enabled, all LLM calls are logged to LangSmith dashboard
    LANGCHAIN_API_KEY: str = os.getenv("LANGCHAIN_API_KEY", "")
    LANGCHAIN_TRACING_V2: str = os.getenv("LANGCHAIN_TRACING_V2", "false")
    
    # --- Application Paths ---
    APP_NAME: str = os.getenv("APP_NAME", "AI-SDLC Accelerator")
    
    # OUTPUT_DIR: where all generated code files are saved
    # Structure: output/api/, output/tests/, output/devops/, output/planning/
    OUTPUT_DIR: str = os.getenv("OUTPUT_DIR", "output")
    
    # GUIDELINES_PATH: the main company standard guideline document
    # This is the DEFAULT file used by the Scrum Master and BA Agent
    # (Developer/Tester/DevOps each have their own in guidelines/)
    GUIDELINES_PATH: str = os.getenv("GUIDELINES_PATH", "guideline.txt")
    
    # DATABASE_PATH: path to the SQLite database file
    DATABASE_PATH: str = os.getenv("DATABASE_PATH", "database/sdlc.db")


def get_settings() -> Settings:
    """
    Factory function — returns a Settings instance.
    
    Used everywhere in the project:
      agent = DeveloperAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)
    """
    return Settings()


# --- Let's verify it works ---
settings = get_settings()
print(f"✅ Settings loaded!")
print(f"   Model: {settings.OPENAI_MODEL}")
print(f"   Output dir: {settings.OUTPUT_DIR}")
print(f"   Guidelines: {settings.GUIDELINES_PATH}")
print(f"   Database: {settings.DATABASE_PATH}")
print(f"   GitHub: {settings.GITHUB_REPO_OWNER}/{settings.GITHUB_REPO_NAME}" 
      if settings.GITHUB_REPO_OWNER else "   GitHub: Not configured")
print(f"   API Key: {'✅ Set' if settings.OPENAI_API_KEY else '❌ Missing!'}")

✅ Settings loaded!
   Model: gpt-4o
   Output dir: output
   Guidelines: guideline.txt
   Database: database/sdlc.db
   GitHub: Shubhankitsirvaiya/AI_SDLC_Test_Repo
   API Key: ✅ Set


---
# 🗄️ SECTION 3: Database Manager (database/db_manager.py)
---
The `DatabaseManager` class manages a SQLite database with **3 pre-seeded tables**:
- **products** (20 rows) — Electronics, accessories, audio, furniture
- **customers** (10 rows) — Sample customer records
- **orders** (25 rows) — Orders linking customers to products

**Why is this important?**  
The BA Agent uses `get_schema_summary()` to show the LLM what data already exists — so it can generate code that works with the real database schema.

In [241]:
# ============================================================================
# STEP 3.1: Database Manager Class  (database/db_manager.py)
# ============================================================================
#
# WHAT THIS DOES:
# ---------------
# Creates and manages a SQLite database with 3 tables:
#   - products  (20 rows) — sample product catalog
#   - customers (10 rows) — sample customer records
#   - orders    (25 rows) — links customers to products
#
# WHY DO WE NEED A DATABASE?
# --------------------------
# The BA Agent needs to know what data ALREADY EXISTS so it can suggest
# APIs that work with real tables. When a user says "I want a product API",
# the BA sees the products table and knows the exact columns, types, and sample data.
#
# HOW IT FITS IN THE PIPELINE:
# ---------------------------
# 1. DatabaseManager is created at startup
# 2. get_schema_summary() generates a Markdown description of all tables
# 3. This schema string is injected into:
#    - BA Agent's system prompt (so it knows what data exists)
#    - Developer Agent's guidelines (so it generates code matching the schema)
#
# FLOW:
#   DatabaseManager → get_schema_summary() → injected into agent prompts
#                                              ↓
#                                    LLM knows the real DB structure


class DatabaseManager:
    """
    Manages the local SQLite database for development and testing.
    
    Auto-creates tables and seeds with sample data on first run.
    Provides query methods and schema introspection for LLM context.
    """

    def __init__(self, db_path: str = "database/sdlc.db"):
        self.db_path = db_path
        # Ensure the directory exists (e.g., database/ folder)
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        # Create tables and seed data on initialization
        self._initialize()

    def _initialize(self):
        """
        Create tables (if they don't exist) and seed with sample data (if empty).
        
        This runs every time DatabaseManager is instantiated, but the
        IF NOT EXISTS and COUNT(*) checks make it idempotent — safe to call repeatedly.
        """
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row  # Allows accessing columns by name: row["name"]
        cursor = conn.cursor()

        # ── TABLE 1: products ──
        # Stores product catalog with pricing, categories, and inventory.
        # This is the main table the Developer Agent will create CRUD endpoints for.
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS products (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                description TEXT,
                price REAL NOT NULL,
                category TEXT,
                stock_quantity INTEGER DEFAULT 0,
                is_active INTEGER DEFAULT 1,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # ── TABLE 2: customers ──
        # Stores customer information. Note the UNIQUE constraint on email —
        # the Developer Agent should enforce this in the generated API code.
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                first_name TEXT NOT NULL,
                last_name TEXT NOT NULL,
                email TEXT UNIQUE NOT NULL,
                phone TEXT,
                is_active INTEGER DEFAULT 1,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # ── TABLE 3: orders ──
        # Junction table linking customers to products.
        # Has FOREIGN KEY constraints — the generated API should handle these
        # (e.g., validate that customer_id and product_id exist before creating an order).
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                customer_id INTEGER NOT NULL,
                product_id INTEGER NOT NULL,
                quantity INTEGER NOT NULL DEFAULT 1,
                total_price REAL NOT NULL,
                status TEXT DEFAULT 'pending',
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (customer_id) REFERENCES customers(id),
                FOREIGN KEY (product_id) REFERENCES products(id)
            )
        """)

        # ── Seed products (only if table is empty) ──
        # These 20 products cover various categories so the BA Agent
        # can suggest relevant endpoints (e.g., "filter by category")
        if cursor.execute("SELECT COUNT(*) FROM products").fetchone()[0] == 0:
            products = [
                ("Laptop Pro 15", "High-performance laptop with 16GB RAM and 512GB SSD", 1299.99, "Electronics", 50),
                ("Wireless Mouse", "Ergonomic wireless mouse with USB-C receiver", 49.99, "Accessories", 200),
                ("Mechanical Keyboard", "RGB mechanical keyboard with Cherry MX Blue switches", 129.99, "Accessories", 150),
                ("4K Monitor 27\"", "Ultra HD IPS monitor, 144Hz, HDR600", 449.99, "Electronics", 75),
                ("USB-C Hub", "7-in-1 USB-C docking station with HDMI, Ethernet, SD reader", 79.99, "Accessories", 300),
                ("Noise Cancelling Headphones", "Over-ear ANC headphones, 30hr battery", 299.99, "Audio", 120),
                ("Webcam HD 1080p", "Full HD webcam with auto-focus and ring light", 89.99, "Electronics", 180),
                ("Standing Desk Mat", "Anti-fatigue standing desk mat, 20x36 inches", 59.99, "Furniture", 90),
                ("Cable Management Kit", "Complete cable organizer set with 50 velcro ties", 24.99, "Accessories", 500),
                ("Desk Lamp LED", "Adjustable LED desk lamp, 5 colour modes, USB charging", 39.99, "Furniture", 160),
                ("Laptop Stand", "Aluminium adjustable laptop riser, fits 10-17 inch", 69.99, "Accessories", 220),
                ("Bluetooth Speaker", "Portable waterproof speaker, 12hr battery, IPX7", 79.99, "Audio", 95),
                ("Smartphone Dock", "Wireless charging dock with MagSafe support", 44.99, "Accessories", 180),
                ("External SSD 1TB", "Portable NVMe SSD, 1050MB/s read speed", 109.99, "Electronics", 130),
                ("Office Chair Cushion", "Memory foam seat cushion, ergonomic design", 34.99, "Furniture", 250),
                ("Webcam Light Bar", "LED monitor light bar, auto-dimming sensor", 54.99, "Accessories", 140),
                ("Tablet Stylus Pro", "Pressure-sensitive stylus pen, 4096 levels", 69.99, "Accessories", 85),
                ("Gaming Mousepad XL", "Extended desk mat, 900x400mm, waterproof surface", 29.99, "Accessories", 310),
                ("Smart Power Strip", "WiFi-enabled power strip, 4 outlets, 3 USB ports", 39.99, "Electronics", 175),
                ("Desk Organiser", "Bamboo multi-slot desk organiser with phone holder", 27.99, "Furniture", 200),
            ]
            cursor.executemany(
                "INSERT INTO products (name, description, price, category, stock_quantity) VALUES (?, ?, ?, ?, ?)",
                products,
            )

        # ── Seed customers (10 sample records) ──
        if cursor.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
            customers = [
                ("Alice", "Johnson", "alice.johnson@example.com", "+1-555-0101"),
                ("Bob", "Smith", "bob.smith@example.com", "+1-555-0102"),
                ("Charlie", "Brown", "charlie.brown@example.com", "+1-555-0103"),
                ("Diana", "Prince", "diana.prince@example.com", "+1-555-0104"),
                ("Edward", "Norton", "edward.norton@example.com", "+1-555-0105"),
                ("Fiona", "Green", "fiona.green@example.com", "+1-555-0106"),
                ("George", "Harrison", "george.harrison@example.com", "+1-555-0107"),
                ("Hannah", "Lee", "hannah.lee@example.com", "+1-555-0108"),
                ("Ivan", "Petrov", "ivan.petrov@example.com", "+1-555-0109"),
                ("Julia", "Martinez", "julia.martinez@example.com", "+1-555-0110"),
            ]
            cursor.executemany(
                "INSERT INTO customers (first_name, last_name, email, phone) VALUES (?, ?, ?, ?)",
                customers,
            )

        # ── Seed orders (25 sample records linking customers ↔ products) ──
        if cursor.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 0:
            orders = [
                (1, 1, 1, 1299.99, "completed"), (1, 3, 2, 259.98, "completed"),
                (2, 2, 1, 49.99, "shipped"), (3, 6, 1, 299.99, "pending"),
                (4, 4, 1, 449.99, "processing"), (5, 5, 3, 239.97, "completed"),
                (2, 7, 1, 89.99, "pending"), (3, 10, 2, 79.98, "shipped"),
                (1, 14, 1, 109.99, "completed"), (6, 12, 2, 159.98, "shipped"),
                (7, 1, 1, 1299.99, "processing"), (8, 11, 1, 69.99, "completed"),
                (9, 15, 3, 104.97, "pending"), (10, 8, 1, 59.99, "shipped"),
                (4, 19, 2, 79.98, "completed"), (5, 16, 1, 54.99, "processing"),
                (6, 20, 1, 27.99, "completed"), (7, 9, 5, 124.95, "shipped"),
                (8, 18, 1, 29.99, "pending"), (2, 13, 1, 44.99, "completed"),
                (9, 4, 1, 449.99, "processing"), (10, 17, 2, 139.98, "completed"),
                (3, 11, 1, 69.99, "shipped"), (1, 20, 2, 55.98, "completed"),
                (6, 3, 1, 129.99, "pending"),
            ]
            cursor.executemany(
                "INSERT INTO orders (customer_id, product_id, quantity, total_price, status) VALUES (?, ?, ?, ?, ?)",
                orders,
            )

        conn.commit()
        conn.close()

    def get_connection(self) -> sqlite3.Connection:
        """Get a database connection with Row factory enabled."""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row  # Access columns by name: row["column"]
        return conn

    def execute_query(self, query: str, params: tuple = ()) -> list:
        """
        Execute a SELECT query and return results as list of dicts.
        Used for ad-hoc queries and in the Streamlit dashboard.
        """
        conn = self.get_connection()
        try:
            cursor = conn.cursor()
            cursor.execute(query, params)
            return [dict(row) for row in cursor.fetchall()]
        finally:
            conn.close()

    def get_table_info(self) -> dict:
        """
        Return schema metadata for all tables.
        Returns: {"products": {"columns": [...], "row_count": 20}, ...}
        """
        conn = self.get_connection()
        try:
            cursor = conn.cursor()
            # sqlite_master stores metadata about all tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
            tables = [row["name"] for row in cursor.fetchall()]
            info = {}
            for table in tables:
                # PRAGMA table_info returns column details (name, type, nullable, pk)
                cursor.execute(f"PRAGMA table_info({table})")
                columns = [
                    {"name": col["name"], "type": col["type"], 
                     "notnull": bool(col["notnull"]), "pk": bool(col["pk"])}
                    for col in cursor.fetchall()
                ]
                cursor.execute(f"SELECT COUNT(*) as cnt FROM {table}")
                count = cursor.fetchone()["cnt"]
                info[table] = {"columns": columns, "row_count": count}
            return info
        finally:
            conn.close()

    def get_schema_summary(self) -> str:
        """
        Generate a human-readable Markdown schema summary with sample data.
        
        THIS IS THE KEY METHOD FOR THE PIPELINE.
        
        What it produces:
          - Table names and row counts
          - Column details (name, type, nullable, primary key)
          - 3 sample rows per table so the LLM sees real data
        
        This output gets injected into:
          1. BA Agent system prompt → so it knows what data exists
          2. Developer Agent guidelines → so generated code matches the schema
        
        Example output:
          ## Database Schema & Sample Data
          ### Table: `products` (20 rows)
          | Column | Type | Nullable | PK |
          | id | INTEGER | NO | ✓ |
          | name | TEXT | NO |  |
          ...
        """
        conn = self.get_connection()
        try:
            cursor = conn.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
            tables = [row["name"] for row in cursor.fetchall()]
            parts = ["## Database Schema & Sample Data\n"]
            for table in tables:
                cursor.execute(f"PRAGMA table_info({table})")
                cols = cursor.fetchall()
                cursor.execute(f"SELECT COUNT(*) as cnt FROM {table}")
                count = cursor.fetchone()["cnt"]
                parts.append(f"### Table: `{table}` ({count} rows)")
                parts.append("| Column | Type | Nullable | PK |")
                parts.append("|--------|------|----------|----|")
                col_names = []
                for c in cols:
                    nullable = "YES" if not c["notnull"] else "NO"
                    pk = "✓" if c["pk"] else ""
                    parts.append(f"| {c['name']} | {c['type']} | {nullable} | {pk} |")
                    col_names.append(c["name"])
                # Show 3 sample rows — this helps the LLM understand the data format
                cursor.execute(f"SELECT * FROM {table} LIMIT 3")
                rows = cursor.fetchall()
                if rows:
                    parts.append(f"\n**Sample rows ({min(3, count)} of {count}):**")
                    parts.append("| " + " | ".join(col_names) + " |")
                    parts.append("| " + " | ".join("---" for _ in col_names) + " |")
                    for row in rows:
                        vals = [str(row[c]) if row[c] is not None else "NULL" for c in col_names]
                        parts.append("| " + " | ".join(vals) + " |")
                parts.append("")
            return "\n".join(parts)
        finally:
            conn.close()


print("✅ DatabaseManager class defined!")

✅ DatabaseManager class defined!


In [242]:
# ============================================================================
# STEP 3.2: Test the Database
# ============================================================================
#
# WHAT'S HAPPENING:
# ----------------
# We create the DatabaseManager, which automatically:
#   1. Creates the SQLite file at database/sdlc.db
#   2. Creates 3 tables (products, customers, orders)
#   3. Seeds them with sample data (20 + 10 + 25 rows)
#
# Then we verify everything is working by:
#   - Listing tables and row counts
#   - Running a sample query
#   - Previewing the schema summary (what the LLM will see)

db = DatabaseManager(db_path=settings.DATABASE_PATH)

# Show what tables exist and how many rows each has
print("📊 Database Tables:")
for table, info in db.get_table_info().items():
    print(f"   {table}: {info['row_count']} rows, {len(info['columns'])} columns")

# Run a sample query — this is how the app would query the DB
print("\n📦 Sample Products:")
products = db.execute_query("SELECT id, name, price, category FROM products LIMIT 5")
for p in products:
    print(f"   [{p['id']}] {p['name']} — ${p['price']} ({p['category']})")

# Preview the schema summary — THIS is what gets injected into agent prompts
# The BA Agent and Developer Agent see this so they know the real DB structure
print("\n" + "="*60)
print("📋 Schema Summary (this Markdown gets sent to the LLM):")
print("="*60)
print(db.get_schema_summary()[:1000] + "\n... (truncated)")

📊 Database Tables:
   products: 20 rows, 9 columns
   customers: 10 rows, 8 columns
   orders: 25 rows, 8 columns

📦 Sample Products:
   [1] Laptop Pro 15 — $1299.99 (Electronics)
   [2] Wireless Mouse — $49.99 (Accessories)
   [3] Mechanical Keyboard — $129.99 (Accessories)
   [4] 4K Monitor 27" — $449.99 (Electronics)
   [5] USB-C Hub — $79.99 (Accessories)

📋 Schema Summary (this Markdown gets sent to the LLM):
## Database Schema & Sample Data

### Table: `products` (20 rows)
| Column | Type | Nullable | PK |
|--------|------|----------|----|
| id | INTEGER | YES | ✓ |
| name | TEXT | NO |  |
| description | TEXT | YES |  |
| price | REAL | NO |  |
| category | TEXT | YES |  |
| stock_quantity | INTEGER | YES |  |
| is_active | INTEGER | YES |  |
| created_at | TIMESTAMP | YES |  |
| updated_at | TIMESTAMP | YES |  |

**Sample rows (3 of 20):**
| id | name | description | price | category | stock_quantity | is_active | created_at | updated_at |
| --- | --- | --- | --- | --- | --- | 

---
# 🔍 SECTION 4: RAG System (rag/loader.py + rag/retriever.py)
---
RAG = **Retrieval Augmented Generation** — instead of stuffing the entire guidelines document into every prompt, we:

1. **Load** the guideline file → split into chunks (800 chars each)
2. **Embed** each chunk using OpenAI Embeddings → store in FAISS vector index
3. **Retrieve** — when an agent needs context, we search the index for the most relevant chunks
4. **Rerank** — a CrossEncoder model re-scores the candidates for better precision

**Each agent has its own guideline file and FAISS index:**
- BA Agent → `guidelines/ba_guidelines.txt`
- Developer → `guidelines/developer_guidelines.txt`
- Tester → `guidelines/tester_guidelines.txt`
- DevOps → `guidelines/devops_guidelines.txt`
- Scrum Master → `guideline.txt` (main doc)

In [243]:
# ============================================================================
# STEP 4.1: Document Loader & Chunker  (rag/loader.py)
# ============================================================================
#
# WHAT IS RAG (Retrieval Augmented Generation)?
# ---------------------------------------------
# Instead of sending the ENTIRE 50-page guideline document with every LLM call:
#   1. We SPLIT the document into small chunks (~800 chars each)
#   2. We EMBED each chunk as a vector (list of 1536 numbers)
#   3. We STORE vectors in a FAISS index (fast similarity search)
#   4. At runtime, we SEARCH for chunks relevant to the query
#   5. We send ONLY the relevant chunks to the LLM — saves tokens and improves focus
#
# THIS FUNCTION handles step 1: Load a .txt file and split it into chunks.
#
# WHY CHUNK OVERLAP?
# ------------------
# If a chunk ends mid-sentence, the next chunk starts with the last 150 chars
# of the previous chunk. This ensures no information is lost at boundaries.
#
# EXAMPLE:
#   Original: "FastAPI uses Pydantic v2 for schema validation. Always use..."
#   Chunk 1:  "FastAPI uses Pydantic v2 for schema validation."
#   Chunk 2:  "for schema validation. Always use..."  ← overlaps by 150 chars


def load_and_chunk_guidelines(
    file_path: str = "guideline.txt",     # The guideline file to process
    chunk_size: int = 800,                # Max chars per chunk
    chunk_overlap: int = 150,             # Overlap between consecutive chunks
) -> list[Document]:
    """
    Load a guideline .txt file and split it into semantically meaningful chunks.
    
    Returns a list of LangChain Document objects, each containing:
      - page_content: the text of that chunk
      - metadata: {"source": "guideline.txt"} — tracks which file it came from
    """
    
    # TextLoader reads the .txt file and wraps it as a LangChain Document
    # encoding="utf-8" handles special characters properly
    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()  # Returns [Document(page_content="full text...", metadata={...})]

    # RecursiveCharacterTextSplitter tries to split at NATURAL BOUNDARIES:
    # It tries each separator in order — first at "## " (section headers),
    # then "### " (subsections), then paragraphs ("\n\n"), then lines ("\n"),
    # then sentences (". "), then words (" "), and finally characters ("").
    # This preserves semantic meaning within each chunk.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],  # Priority order
        keep_separator=True,  # Keep the separator in the chunk (e.g., "## " stays)
    )
    
    # Split the single large document into many small chunks
    chunks = splitter.split_documents(documents)
    return chunks


# # Test it (uncomment to verify):
# chunks = load_and_chunk_guidelines("guideline.txt")
# print(f"✅ Loaded and chunked guideline.txt → {len(chunks)} chunks")
# print(f"   First chunk preview ({len(chunks[0].page_content)} chars):")
# print(f"   '{chunks[0].page_content[:150]}...'")


In [244]:
# ============================================================================
# STEP 4.2: Reranker + GuidelineRetriever  (rag/retriever.py)
# ============================================================================
#
# THE TWO-STAGE RAG PIPELINE:
# ---------------------------
# Stage 1: FAISS Similarity Search (FAST but APPROXIMATE)
#   - Converts query to a vector using OpenAI Embeddings
#   - Finds 15 nearest vectors in the FAISS index (cosine similarity)
#   - Fast because it's just math (dot products), but can miss nuance
#
# Stage 2: CrossEncoder Reranking (SLOW but PRECISE)
#   - Takes the 15 FAISS candidates
#   - Feeds each (query, document) pair through a neural network together
#   - The model reads BOTH texts simultaneously → understands context better
#   - Re-scores and picks the top 5 most relevant
#
# WHY TWO STAGES?
#   FAISS alone might return "FastAPI routing" when you asked about "API routes"
#   because the embeddings are similar but the content might not be the best match.
#   The CrossEncoder catches these cases by reading the actual text.
#
# ANALOGY:
#   Stage 1 (FAISS) = looking at book covers to find relevant books (fast scan)
#   Stage 2 (Reranker) = reading the first page of each book to pick the best ones


class Reranker:
    """
    Cross-encoder reranker — scores (query, document) pairs for relevance.
    
    Model: ms-marco-MiniLM-L-6-v2 (a small, fast cross-encoder)
    Input: a query string + a list of Document objects
    Output: the top_k most relevant Documents, sorted by score
    """
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        # Load the cross-encoder model (downloads ~80MB on first run)
        self.model = CrossEncoder(model_name)

    def rerank(self, query: str, documents: list[Document], top_k: int = 5) -> list[Document]:
        """Score each (query, doc) pair and return the top_k most relevant."""
        if not documents:
            return []
        
        # Create (query, document_text) pairs for the cross-encoder
        # Example: [("FastAPI routes", "chunk about routing..."), ("FastAPI routes", "chunk about auth...")]
        pairs = [(query, doc.page_content) for doc in documents]
        
        # The model scores each pair — higher score = more relevant
        # Returns a numpy array of floats, e.g., [0.85, 0.23, 0.91, ...]
        scores = self.model.predict(pairs)
        
        # Sort by score descending, take top_k
        scored = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored[:top_k]]


class GuidelineRetriever:
    """
    Complete RAG retriever: FAISS vector store + CrossEncoder reranking.
    
    LIFECYCLE:
    1. __init__() — stores config but doesn't build anything yet (lazy)
    2. initialize() — builds FAISS index from chunks + loads reranker (called ONCE)
    3. retrieve(query) — searches index → reranks → returns relevant text
    
    WHY LAZY INITIALIZATION?
    Each guideline file gets its own retriever instance. We don't want to build
    5 FAISS indexes at startup if only 2 agents will run. So we build on first use.
    """
    def __init__(self, guidelines_path: str = "guideline.txt", api_key: str = ""):
        self.guidelines_path = guidelines_path   # Path to the .txt file
        self.api_key = api_key                   # OpenAI API key for embeddings
        self.vectorstore = None                  # FAISS index (built on first use)
        self.reranker = None                     # CrossEncoder (loaded on first use)
        self._initialized = False                # Flag to prevent re-initialization

    def initialize(self):
        """
        Build the FAISS vector store and reranker.
        Called ONCE on first retrieve() call, then cached.
        
        Steps:
        1. Load the guideline file → split into ~800-char chunks
        2. Create OpenAI embeddings (each chunk → 1536-dimensional vector)
        3. Build FAISS index from chunks + their embeddings
        4. Initialize the CrossEncoder reranker model
        """
        if self._initialized:
            return  # Already built — skip
        
        # Step 1: Load and chunk the guideline file
        chunks = load_and_chunk_guidelines(self.guidelines_path)
        
        # Step 2: Create embedding model
        # OpenAI's text-embedding-ada-002 converts text → 1536-dim vector
        # Cost: ~$0.0001 per 1000 tokens (very cheap)
        embeddings = OpenAIEmbeddings(api_key=self.api_key)
        
        # Step 3: Build FAISS index
        # FAISS.from_documents() does three things:
        #   a) Embeds each chunk using the embedding model
        #   b) Stores the vectors in a FAISS index (for fast similarity search)
        #   c) Maintains a mapping from vector → original Document
        self.vectorstore = FAISS.from_documents(chunks, embeddings)
        
        # Step 4: Initialize the CrossEncoder reranker
        self.reranker = Reranker()
        self._initialized = True

    def retrieve(self, query: str, top_k: int = 5, initial_k: int = 15) -> str:
        """
        Retrieve relevant guideline sections using the 2-stage pipeline.
        
        Args:
            query: What we're looking for
                   e.g., "FastAPI code generation database models"
            top_k: How many final results to return (default 5)
            initial_k: How many FAISS candidates to consider (default 15)
        
        Returns:
            A single string with the top_k most relevant chunks concatenated,
            separated by "---". This gets injected directly into agent prompts.
        
        Flow:
            query → FAISS (15 candidates) → CrossEncoder (top 5) → joined text
        """
        self.initialize()  # Build index on first call (lazy init)
        
        # Stage 1: FAISS broad retrieval
        # Converts query to vector → finds 15 nearest vectors in the index
        candidates = self.vectorstore.similarity_search(query, k=initial_k)
        
        # Stage 2: CrossEncoder reranking
        # Reads each (query, candidate) pair together → returns top 5
        reranked = self.reranker.rerank(query, candidates, top_k=top_k)
        
        # Join the top chunks into a single string with separators
        return "\n\n---\n\n".join(doc.page_content for doc in reranked)

    def retrieve_all(self) -> str:
        """Return the FULL guideline document (no chunking/search). For debugging."""
        with open(self.guidelines_path, "r", encoding="utf-8") as f:
            return f.read()


print("✅ Reranker and GuidelineRetriever classes defined!")

✅ Reranker and GuidelineRetriever classes defined!


In [245]:
# ============================================================================
# STEP 4.3: Per-Agent Retriever Singletons  (rag/retriever.py)
# ============================================================================
#
# WHAT ARE "PER-AGENT RETRIEVERS"?
# --------------------------------
# Each agent has its own guideline .txt file:
#   - guideline.txt                    → BA Agent + Scrum Master (general standards)
#   - guidelines/developer_guidelines.txt → Developer Agent (code patterns)
#   - guidelines/tester_guidelines.txt    → Tester Agent (test patterns)
#   - guidelines/devops_guidelines.txt    → DevOps Agent (infrastructure patterns)
#   - guidelines/ba_guidelines.txt        → BA Agent (conversation rules)
#
# Each file gets its OWN FAISS index (its own set of embedded chunks).
# This means the Developer only sees developer-relevant guidelines,
# the Tester only sees testing guidelines, etc.
#
# WHAT IS THE CACHE?
# ------------------
# _retriever_cache is a dictionary: {"guideline.txt": retriever_instance, ...}
# When get_retriever("guideline.txt") is called:
#   - First time: creates a new GuidelineRetriever and stores it in the cache
#   - Second time: returns the cached instance (no rebuilding the FAISS index)
# This prevents building the same index multiple times (each build costs API calls).
#
# WHAT ARE "INDEXES"?
# -------------------
# An index = a FAISS vector database built from one guideline file.
# Each .txt file is chunked, embedded, and stored in its own FAISS index.
# So we have ~5 separate FAISS indexes, one per guideline file.

_retriever_cache: dict[str, GuidelineRetriever] = {}  # Singleton cache


def get_retriever(guidelines_path: str | None = None) -> GuidelineRetriever:
    """
    Return (and lazily create) a GuidelineRetriever for the given file.
    
    Uses caching — each guideline file gets its own FAISS index, built only once.
    If no path given, returns the default retriever (guideline.txt).
    
    Example usage:
        retriever = get_retriever("guidelines/developer_guidelines.txt")
        context = retriever.retrieve("FastAPI models schemas", top_k=8)
    """
    path = guidelines_path or settings.GUIDELINES_PATH  # Default: "guideline.txt"
    if path not in _retriever_cache:
        # First time for this file — create a new retriever
        # (FAISS index will be built lazily on first .retrieve() call)
        _retriever_cache[path] = GuidelineRetriever(
            guidelines_path=path, api_key=settings.OPENAI_API_KEY,
        )
    return _retriever_cache[path]


# ── Convenience functions for each agent ──────────────────────────
# These make the code cleaner — instead of passing file paths everywhere,
# each agent just calls get_developer_retriever() etc.

def get_developer_retriever() -> GuidelineRetriever:
    """Developer Agent's retriever — guidelines about FastAPI, SQLAlchemy, code patterns."""
    return get_retriever("guidelines/developer_guidelines.txt")

def get_tester_retriever() -> GuidelineRetriever:
    """Tester Agent's retriever — guidelines about pytest, test patterns, fixtures."""
    return get_retriever("guidelines/tester_guidelines.txt")

def get_devops_retriever() -> GuidelineRetriever:
    """DevOps Agent's retriever — guidelines about Docker, K8s, CI/CD, GitHub Actions."""
    return get_retriever("guidelines/devops_guidelines.txt")

def get_ba_retriever() -> GuidelineRetriever:
    """BA Agent's retriever — guidelines about conversation rules, spec format."""
    return get_retriever("guidelines/ba_guidelines.txt")


# ── Test the retriever ──
# This builds the FAISS index for guideline.txt on first run (calls OpenAI Embeddings API)
print("🔍 Testing RAG retrieval...")
retriever = get_retriever()
result = retriever.retrieve("API design response format authentication", top_k=3)
print(f"✅ Retrieved {len(result)} chars of relevant guidelines")
print(f"   Preview: '{result[:200]}...'")

🔍 Testing RAG retrieval...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6315.18it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Retrieved 1999 chars of relevant guidelines
   Preview: '## 3. API DESIGN STANDARDS

### 3.1 URL Convention
- Use lowercase with hyphens: `/api/v1/user-profiles`
- Always version APIs: `/api/v1/...`
- Use plural nouns for resources: `/users`, `/orders`
- Ne...'


---
# 🛠️ SECTION 5: Utility Functions (utils/)
---
Three helper modules:
1. **parsers.py** — Robust JSON parsing with LLM auto-repair (3-tier fallback)
2. **file_utils.py** — Save/read/list generated artifacts on disk
3. **git_utils.py** — Git init, branch, commit, push, and GitHub PR creation

In [246]:
# ============================================================================
# STEP 5.1: JSON Parser with Self-Healing  (utils/parsers.py)
# ============================================================================
#
# THE PROBLEM:
# -----------
# Every agent returns JSON from the LLM (code files, planning docs, configs).
# But LLMs sometimes return BROKEN JSON:
#   - Extra commas: {"a": 1, "b": 2,}  ← trailing comma is invalid JSON
#   - Wrong quotes: {'a': 1}  ← single quotes instead of double quotes
#   - Markdown fences: ```json {"a": 1} ```  ← code block wrapping
#   - Truncated output: {"a": 1, "b":  ← model hit token limit
#
# THE SOLUTION — 3-TIER FALLBACK:
# --------------------------------
# Tier 1: json.loads() — fastest, works 90% of the time
# Tier 2: LangChain's JsonOutputParser — handles some edge cases (e.g., single quotes)
# Tier 3: Ask the LLM itself to FIX the JSON — surprisingly effective!
#          "Hey GPT, you gave me broken JSON. Fix it."
#
# This function is used by EVERY agent to safely parse LLM responses.
# Without it, one bad JSON response would crash the entire pipeline.


def parse_llm_json(raw: str, llm) -> dict:
    """
    Parse JSON from an LLM response with automatic repair on failure.
    
    Args:
        raw: The raw string response from the LLM (may contain markdown fences)
        llm: The ChatOpenAI instance (used for Tier 3 self-repair)
    
    Returns:
        Parsed JSON as a Python dict
    
    Used by: BAAgent.extract_spec(), ScrumMasterAgent.generate_planning(),
             DeveloperAgent.generate_code(), TesterAgent.generate_tests(),
             DevOpsAgent.generate_configs()
    """
    text = raw.strip()

    # Pre-processing: Strip markdown code fences (```json ... ```)
    # LLMs LOVE to wrap JSON in code blocks even when told not to
    # Before: ```json\n{"a": 1}\n```
    # After:  {"a": 1}
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*\n?", "", text)  # Remove opening ```json
        text = re.sub(r"\n?```\s*$", "", text)             # Remove closing ```

    # ── Tier 1: Direct parse (fastest — works ~90% of the time) ──
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("⚠️  [Parser] Direct JSON parse failed — trying LangChain parser...")

    # ── Tier 2: LangChain's JsonOutputParser ──
    # This handles some edge cases like single-quoted keys, trailing commas, etc.
    parser = JsonOutputParser()
    try:
        return parser.parse(text)
    except Exception:
        pass

    # ── Tier 3: LLM Self-Repair ──
    # We send the broken JSON back to the LLM and ask it to fix it.
    # This is surprisingly effective — the LLM can usually spot and fix its own mistakes.
    # Cost: one extra API call (~$0.01), but prevents pipeline crash.
    print("⚠️  [Parser] LangChain parser also failed — asking LLM to repair JSON...")
    fix_prompt = (
        "The following text was supposed to be valid JSON but is malformed. "
        "Return ONLY the corrected JSON with no explanation or markdown:\n\n"
        + raw
    )
    response = llm.invoke([HumanMessage(content=fix_prompt)])
    fixed = response.content.strip()
    if fixed.startswith("```"):
        fixed = re.sub(r"^```(?:json)?\s*\n?", "", fixed)
        fixed = re.sub(r"\n?```\s*$", "", fixed)
    return json.loads(fixed)  # If this fails too, the exception propagates up


print("✅ parse_llm_json function defined!")

✅ parse_llm_json function defined!


In [247]:
# ============================================================================
# STEP 5.2: File Utilities  (utils/file_utils.py)
# ============================================================================
#
# WHAT THESE DO:
# -------------
# After each agent generates code/configs, we need to SAVE them to disk.
# These functions handle the file I/O operations.
#
# WHERE FILES GO (the output directory structure):
# ------------------------------------------------
#   output/
#   ├── api/                    ← Developer Agent's generated code
#   │   ├── app/main.py
#   │   ├── app/models/product.py
#   │   ├── app/routes/product.py
#   │   └── requirements.txt
#   ├── tests/                  ← Tester Agent's generated tests
#   │   ├── conftest.py
#   │   └── test_product.py
#   ├── devops/                 ← DevOps Agent's configs
#   │   ├── Dockerfile
#   │   ├── docker-compose.yml
#   │   └── k8s/deployment.yaml
#   └── planning/               ← Scrum Master's planning docs
#       ├── jira_tickets.md
#       ├── confluence_doc.md
#       └── readme.md
#
# WHO CALLS THESE:
# ----------------
# - TesterAgent.testing_node() → saves api/ + tests/ before running pytest
# - DevOpsAgent.devops_node() → saves ALL artefacts (api + tests + devops + planning)
# - Streamlit UI → list_artifacts() to show the generated files in the dashboard


def save_artifacts(output_dir: str, artifacts: dict[str, dict[str, str]]):
    """
    Save generated artifacts to the output directory.
    
    Args:
        output_dir: e.g., "output"
        artifacts: nested dict where:
            - outer key = category ("api", "tests", "devops", "planning")
            - inner key = filename (can include subfolders like "app/main.py")
            - value = file content as string
    
    Example:
        save_artifacts("output", {
            "api": {"app/main.py": "from fastapi import...", "requirements.txt": "..."},
            "tests": {"conftest.py": "import pytest...", "test_product.py": "..."},
        })
    Creates: output/api/app/main.py, output/tests/conftest.py, etc.
    """
    for category, files in artifacts.items():
        category_dir = os.path.join(output_dir, category)
        os.makedirs(category_dir, exist_ok=True)
        for filename, content in files.items():
            filepath = os.path.join(category_dir, filename)
            # Handle nested paths — creates subdirectories as needed
            # e.g., ".github/workflows/ci-cd.yml" → creates .github/workflows/
            os.makedirs(os.path.dirname(filepath), exist_ok=True)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)


def read_artifact(output_dir: str, category: str, filename: str) -> str:
    """Read a single artifact file. Used by the Streamlit UI to display code."""
    filepath = os.path.join(output_dir, category, filename)
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    return ""


def list_artifacts(output_dir: str) -> dict[str, list[str]]:
    """
    List all generated artifact files grouped by category.
    
    Returns: {"api": ["app/main.py", "app/models.py"], "tests": ["test_product.py"], ...}
    Used by: Streamlit dashboard tab to show what was generated.
    """
    result = {}
    if not os.path.exists(output_dir):
        return result
    for category in os.listdir(output_dir):
        cat_path = os.path.join(output_dir, category)
        if os.path.isdir(cat_path):
            files = []
            for root, _, filenames in os.walk(cat_path):
                for fn in filenames:
                    if fn == ".gitkeep":
                        continue
                    rel = os.path.relpath(os.path.join(root, fn), cat_path)
                    files.append(rel)
            if files:
                result[category] = sorted(files)
    return result


print("✅ File utility functions defined!")

✅ File utility functions defined!


In [248]:
# ============================================================================
# STEP 5.3: Git & GitHub PR Utilities  (utils/git_utils.py)
# ============================================================================
#
# WHAT THESE DO:
# -------------
# After the DevOps Agent generates all configs, we want to:
#   1. Initialize a git repo in the output directory
#   2. Create a feature branch (e.g., "feature/library-api-stub")
#   3. Commit all generated files
#   4. Push to GitHub
#   5. Create a Pull Request via GitHub's REST API
#
# This automates the entire "create PR" workflow — no manual git commands needed.
#
# WHO CALLS THESE:
# ----------------
# Only the DevOps Agent (devops_node) calls these functions.
# It's the LAST agent in the pipeline, so all code is already generated.
#
# OPTIONAL: If GITHUB_TOKEN is not set, the PR step is skipped silently.


def _sanitize_branch_name(name: str) -> str:
    """
    Convert an arbitrary string into a valid git branch name.
    
    Git branch names can't contain spaces, special chars, consecutive dots, etc.
    Example: "Library API v2!" → "Library-API-v2"
    """
    name = name.replace(" ", "-")
    name = re.sub(r"[^a-zA-Z0-9/_\-.]", "", name)  # Keep only safe chars
    name = re.sub(r"\.{2,}", ".", name)               # No consecutive dots (..)
    name = re.sub(r"-{2,}", "-", name)                 # No consecutive dashes (--)
    name = name.strip("-./")
    return name or "feature/generated-stub"


def init_local_repo(project_dir: str) -> str:
    """
    Initialize a local git repo in the project directory.
    
    What happens:
      1. Check if .git/ already exists (skip if so)
      2. Run git init
      3. Stage all files (git add .)
      4. Create initial commit
    
    Requires: gitpython package (pip install gitpython)
    """
    try:
        from git import Repo  # gitpython library
        if not os.path.exists(os.path.join(project_dir, ".git")):
            repo = Repo.init(project_dir)
            # Add all generated files to the index
            repo.index.add([f for f in _walk_files(project_dir) if not f.startswith(".git")])
            repo.index.commit("Initial commit — AI-SDLC generated project")
        return project_dir
    except ImportError:
        # gitpython not installed — skip silently
        return project_dir


def create_branch_and_commit(project_dir: str, branch_name: str, message: str):
    """
    Create a new git branch, stage all changes, and commit.
    
    What happens:
      1. Sanitize branch name (remove invalid chars)
      2. Stage all current changes FIRST (avoids checkout conflicts)
      3. If branch exists → checkout it
         If branch doesn't exist → create and checkout it
      4. Stage ALL files again (git add -A)
      5. Commit with the given message
    """
    branch_name = _sanitize_branch_name(branch_name)
    try:
        from git import Repo
        repo = Repo(project_dir)

        # Stage all current changes FIRST to avoid checkout conflicts
        # Without this, git refuses to switch branches if files have changed
        repo.git.add(A=True)
        if repo.is_dirty(index=True):
            repo.index.commit("WIP — staging changes before branch switch")

        # Switch to branch — create if it doesn't exist
        if branch_name in [b.name for b in repo.branches]:
            repo.git.checkout(branch_name)
        else:
            repo.git.checkout("-b", branch_name)

        # Stage and commit all changes on the target branch
        repo.git.add(A=True)
        if repo.is_dirty(index=True):
            repo.index.commit(message)
    except ImportError:
        pass


def push_and_create_pr(
    project_dir: str, branch_name: str, title: str, body: str,
    github_token: str, repo_owner: str, repo_name: str, base_branch: str = "main",
) -> dict:
    """
    Push branch to GitHub and create a Pull Request via REST API.
    
    Steps:
      1. Set the remote URL (with token embedded for authentication)
      2. git push --set-upstream origin <branch>
      3. POST to https://api.github.com/repos/{owner}/{repo}/pulls
      4. Return the PR URL
    
    Args:
        project_dir: local directory with .git/
        branch_name: the feature branch to push
        title: PR title (e.g., "feat: library-api — AI-generated API stub")
        body: PR description (includes spec JSON + build log)
        github_token: GitHub personal access token
        repo_owner: GitHub username or org
        repo_name: repository name
        base_branch: target branch for the PR (default: "main")
    
    Returns:
        {"success": True, "pr_url": "https://github.com/...", "pr_number": 42}
        or {"success": False, "error": "reason"}
    """
    branch_name = _sanitize_branch_name(branch_name)
    try:
        from git import Repo
        import urllib.request

        repo = Repo(project_dir)
        
        # Set remote URL with embedded token for authentication
        # Format: https://<token>@github.com/<owner>/<repo>.git
        remote_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
        if "origin" in [r.name for r in repo.remotes]:
            repo.delete_remote("origin")  # Remove old remote if exists
        repo.create_remote("origin", remote_url)
        
        # Push the branch to GitHub
        repo.git.push("--set-upstream", "origin", branch_name)

        # Create PR via GitHub REST API (POST request)
        api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/pulls"
        pr_data = json.dumps(
            {"title": title, "body": body, "head": branch_name, "base": base_branch}
        ).encode()
        req = urllib.request.Request(
            api_url, data=pr_data,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github+json",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            result = json.loads(resp.read().decode())
        return {"success": True, "pr_url": result.get("html_url", ""), "pr_number": result.get("number")}
    except Exception as e:
        return {"success": False, "error": str(e)}


def _walk_files(directory: str) -> list[str]:
    """Walk a directory recursively and return relative file paths."""
    files = []
    for root, _, filenames in os.walk(directory):
        for fn in filenames:
            rel = os.path.relpath(os.path.join(root, fn), directory)
            files.append(rel.replace("\\", "/"))
    return files


print("✅ Git utility functions defined!")

✅ Git utility functions defined!


---
# 📝 SECTION 6: Prompt Templates (agents/prompts.py)
---
These are the **8 prompt templates** that drive all agent behavior. Each prompt tells the LLM:
- What role it's playing
- What inputs it has (spec, guidelines, code, etc.)
- What format to return the output in (JSON)

**Key insight:** The prompts ARE the agents' intelligence. The code just orchestrates — the prompts define behavior.

In [249]:
# ============================================================================
# STEP 6.1: BA Agent Prompt Templates  (agents/prompts.py — part 1)
# ============================================================================
#
# WHAT ARE PROMPT TEMPLATES?
# --------------------------
# These are the "job descriptions" we give the LLM. Each agent gets its own
# system prompt telling it:
#   - WHO it is (role)
#   - WHAT it should produce (output format)
#   - HOW to do its job (rules and constraints)
#
# Each prompt has {placeholders} that get filled at runtime with real data:
#   {guidelines_context} → RAG-retrieved company guidelines
#   {db_schema}          → Database schema info
#   {conversation}       → Full chat history (for spec extraction)
#
# WHERE IN THE PIPELINE:
# ----------------------
# The BA Agent runs BEFORE the pipeline starts. It's the user-facing chat
# that gathers requirements and produces the confirmed spec JSON.
#
#   User ↔ BA_CONVERSATION_SYSTEM → multi-turn chat
#   When confirmed → BA_SPEC_EXTRACTION_PROMPT → structured JSON spec
#
# These 2 prompts cover the entire BA workflow.


# ── PROMPT 1 of 8: BA Agent Conversation (system prompt) ──
# This is the SYSTEM MESSAGE for every chat turn. The LLM sees this first,
# then the full conversation history, then the latest user message.
#
# KEY FEATURES:
# - Includes RAG guidelines so the BA follows company standards automatically
# - Includes DB schema so the BA knows what data exists (products, customers, orders)
# - Rules: max 3-4 questions at a time, confirm with [SPEC_CONFIRMED] token
# - The [SPEC_CONFIRMED] token is how the Streamlit UI detects "conversation is done"

BA_CONVERSATION_SYSTEM = """You are the **Business Analyst (BA) Agent** for a spec-driven API development platform.
Your job is to help the user define a complete API specification through conversation.
You are the first agent in the pipeline — you query the database, understand the business need,
and produce a clear technical specification for the downstream agents.

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines_context}

## Available Database Schema & Data
{db_schema}

## Your Responsibilities
1. **First**, review the database schema and sample data above to understand what data already exists.
2. Based on the available data, help the user understand what APIs and features can be built.
3. Understand the API's **purpose and business context**.
4. Identify every **endpoint** (path, HTTP method, request body, response schema, auth).
5. Clarify **data models** and their relationships — reference existing DB tables when applicable.
6. Confirm **authentication / authorisation** requirements.
7. Identify **business rules**, validations, and edge-case behaviour.
8. Capture **non-functional requirements** (rate-limiting, caching, pagination).

## Conversation Rules
- When the user describes a business need, proactively reference the existing database tables and suggest what can be built with the available data.
- Ask focused, numbered questions — no more than 3–4 at a time.
- After gathering enough detail, present a **structured specification summary** in Markdown and explicitly ask the user to confirm.
- If the user confirms, respond with exactly the token `[SPEC_CONFIRMED]` at the very end of your message so the system can detect it.
- Always be professional, concise, and helpful.
- Incorporate company guideline standards (response format, auth type, naming conventions) automatically — inform the user which standards you are applying.
"""


# ── PROMPT 2 of 8: BA Agent Spec Extraction ──
# After the user confirms the spec in the chat, this prompt is used to
# extract a STRUCTURED JSON specification from the entire conversation.
#
# HOW IT WORKS:
# 1. The full conversation (all user + assistant messages) is concatenated
# 2. This prompt tells the LLM: "read this conversation and extract the spec"
# 3. The LLM returns a JSON object with a specific schema
# 4. parse_llm_json() cleans up the output (remove markdown fences, etc.)
#
# The output JSON becomes the INPUT to the entire pipeline:
#   spec → Planning → Developer → Tester → DevOps

BA_SPEC_EXTRACTION_PROMPT = """Based on the full conversation below, extract a detailed API specification as a JSON object.

## Conversation
{conversation}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

Return a valid JSON object (no markdown fences) with this exact schema:
{{
    "project_name": "string",
    "description": "string",
    "base_path": "/api/v1",
    "endpoints": [
        {{
            "path": "string",
            "method": "GET|POST|PUT|PATCH|DELETE",
            "summary": "string",
            "description": "string",
            "request_body": {{}},
            "response_schema": {{}},
            "auth_required": true,
            "query_params": []
        }}
    ],
    "models": [
        {{
            "name": "string",
            "table_name": "string",
            "fields": [
                {{"name": "string", "type": "string", "required": true, "description": "string"}}
            ]
        }}
    ],
    "auth_type": "jwt",
    "database_type": "sqlite",
    "business_rules": ["string"],
    "non_functional": ["string"]
}}
"""

print("✅ BA Agent prompts defined (2/8)")
print("   PROMPT 1: BA_CONVERSATION_SYSTEM — system prompt for multi-turn chat")
print("   PROMPT 2: BA_SPEC_EXTRACTION_PROMPT — extracts JSON spec from conversation")

✅ BA Agent prompts defined (2/8)
   PROMPT 1: BA_CONVERSATION_SYSTEM — system prompt for multi-turn chat
   PROMPT 2: BA_SPEC_EXTRACTION_PROMPT — extracts JSON spec from conversation


In [250]:
# ============================================================================
# STEP 6.2: Scrum Master Prompts — Planning + Orchestrator  (prompts 3 & 4)
# ============================================================================
#
# The Scrum Master is the MOST IMPORTANT agent because it has TWO jobs:
#   Job 1 (Planning): Generate project planning documents
#   Job 2 (Orchestrator): Decide which agent runs next (the "brain")
#
# WHERE IN THE PIPELINE:
# ----------------------
#   [Planning Node] ←── uses PLANNING_PROMPT
#         ↓
#   [Orchestrator Node] ←── uses ORCHESTRATOR_PROMPT (called after EVERY agent)
#         ↓
#   Routes to: developer / tester / devops / end
#
# The orchestrator is called MULTIPLE TIMES during a pipeline run:
#   1. After planning → routes to developer
#   2. After developer → routes to tester
#   3. After tester (pass) → routes to devops
#   3b. After tester (fail) → routes back to developer (fix loop!)
#   4. After devops → routes to end


# ── PROMPT 3 of 8: Planning ──
# Input: The confirmed spec JSON from BA Agent
# Output: JSON with 4 keys: jira_tickets, confluence_doc, flowchart, readme
# These planning docs are saved as markdown files in the output directory.

PLANNING_PROMPT = """You are the **Scrum Master Agent** responsible for project planning.

## Confirmed API Specification
{spec}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

Generate comprehensive planning artefacts as a JSON object with these keys:

1. **jira_tickets** — Markdown string containing an Epic and Stories/Tasks in Jira-style format.
2. **confluence_doc** — Markdown string containing a full technical design document (overview, architecture, data model, API contract, security, deployment).
3. **flowchart** — Markdown string containing a Mermaid flowchart (`graph TD` syntax) showing the API request flow and architecture.
4. **readme** — Markdown string for the project README (setup, usage, architecture diagram, endpoints table).

Return a single valid JSON object (no markdown fences).
"""


# ── PROMPT 4 of 8: Orchestrator ──
# This is the HEART of the hub-and-spoke system.
#
# HOW IT WORKS:
# 1. After each agent finishes, the orchestrator node calls decide_next()
# 2. The LLM sees the FULL pipeline state (what's done, what failed)
# 3. It returns ONE WORD: developer, tester, devops, or end
# 4. LangGraph's conditional edge reads that word and routes accordingly
#
# The {placeholders} give the LLM enough context to make the right decision:
#   current_phase  — "planning_complete", "development_complete", etc.
#   has_api_code   — True/False
#   test_passed    — True/False (key for fix loop!)
#   retry_count    — How many times developer has tried fixing
#   recent_log     — Last 5 build log entries for context
#
# DECISION RULES (defined in the prompt):
#   planning_complete     → developer (generate code)
#   development_complete  → tester (test the code)
#   testing_complete + PASS → devops (deploy)
#   testing_complete + FAIL + retries left → developer (fix code!)
#   testing_complete + FAIL + max retries → devops (best effort)
#   devops_complete       → end (done!)

ORCHESTRATOR_PROMPT = """You are the **Scrum Master Orchestrator** — the central hub of a multi-agent system.

You have access to ALL agents and decide who runs next after every step.
Every agent reports back to YOU, and YOU alone decide the next action.

## Your Agents
- **developer** — Generates or fixes API code
- **tester** — Generates and executes tests
- **devops** — Generates Docker, K8s, CI/CD configs

## Current State
- Current phase: {current_phase}
- Has API code: {has_api_code}
- Has test code: {has_test_code}
- Tests passed: {test_passed}
- Has DevOps config: {has_devops}
- Retry count: {retry_count} / {max_retries}
- Build log (last 5 entries):
{recent_log}

## Decision Rules
1. If current_phase is "planning_complete" → route to "developer" (code needs to be created first)
2. If current_phase is "development_complete" → route to "tester" (code needs testing)
3. If current_phase is "testing_complete":
   a. If tests PASSED → route to "devops" (code is ready for deployment config)
   b. If tests FAILED and retry_count < max_retries → route to "developer" (fix the code)
   c. If tests FAILED and retry_count >= max_retries → route to "devops" (proceed with best effort)
4. If current_phase is "devops_complete" → route to "end" (all work done)

You MUST respond with ONLY one word: developer, tester, devops, or end
"""

print("✅ Scrum Master prompts defined (4/8)")
print("   PROMPT 3: PLANNING_PROMPT — generates Jira, Confluence, flowchart, README")
print("   PROMPT 4: ORCHESTRATOR_PROMPT — the brain that routes between agents")

✅ Scrum Master prompts defined (4/8)
   PROMPT 3: PLANNING_PROMPT — generates Jira, Confluence, flowchart, README
   PROMPT 4: ORCHESTRATOR_PROMPT — the brain that routes between agents


In [251]:
# ============================================================================
# STEP 6.3: Developer + Tester + DevOps Prompts  (prompts 5–8)
# ============================================================================
#
# These 4 prompts cover the remaining 3 agents in the pipeline.
# The Developer has 2 prompts (generate + fix), the rest have 1 each.
#
# PIPELINE CONTEXT — when each prompt is used:
# ─────────────────────────────────────────────
#   Planning ✓
#        ↓
#   Developer (DEVELOPER_PROMPT)     ← PROMPT 5: first code generation
#        ↓
#   Tester (TESTER_PROMPT)           ← PROMPT 7: generate + run tests
#        ↓
#   IF FAIL → Developer (DEVELOPER_FIX_PROMPT)  ← PROMPT 6: fix mode!
#           → Tester (TESTER_PROMPT again)       ← re-test
#           → (up to 3 retries)
#        ↓
#   DevOps (DEVOPS_PROMPT)           ← PROMPT 8: Docker/K8s/CI-CD
#        ↓
#   END


# ── PROMPT 5 of 8: Developer — Generate Code (first run) ──
# Input: spec + planning + guidelines
# Output: JSON object where keys = filenames, values = source code
#
# The prompt specifies the EXACT file structure the LLM must produce:
#   app/main.py, app/config.py, app/database.py, app/models/*, app/schemas/*,
#   app/routes/*, app/services/*, app/dependencies.py, requirements.txt
#
# KEY RULE: "every service function MUST accept db: Session as first parameter"
# This is critical because the Tester creates inline service functions with
# the same signature — if they don't match, tests will fail.

DEVELOPER_PROMPT = """You are the **Developer Agent** responsible for generating production-ready API stub code.

## Confirmed API Specification
{spec}

## Planning
{planning}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

Generate a complete FastAPI project. Return a JSON object where each key is a filename and the value is the full file content.

Required files:
- `app/main.py` — FastAPI app entry-point with CORS, error handling, router includes, and `/health` endpoint.
- `app/config.py` — Settings management using pydantic-settings / env vars.
- `app/database.py` — SQLAlchemy engine + session (async or sync) with SQLite for dev.
- `app/models/<entity>.py` — One file per model with SQLAlchemy ORM classes.
- `app/schemas/<entity>.py` — Pydantic v2 schemas (Create, Update, Response).
- `app/routes/<entity>.py` — Route handlers returning the company-standard envelope format.
- `app/services/<entity>.py` — Business-logic layer. EVERY service function MUST accept `db: Session` as its first parameter.
- `app/dependencies.py` — Shared deps (get_db, get_current_user if auth needed).
- `app/middleware/error_handler.py` — Global exception handler.
- `requirements.txt` — Project runtime dependencies.

Return a single valid JSON object (no markdown fences).
"""


# ── PROMPT 6 of 8: Developer — Fix Code (self-healing loop) ──
# This prompt is ONLY used when tests FAIL and the orchestrator routes
# back to the developer for a fix attempt.
#
# Input: test failure output + current code + spec + guidelines
# Output: CORRECTED JSON object with ALL files (replaces entire codebase)
#
# KEY INSIGHT: The prompt tells the developer that tests are SELF-CONTAINED.
# Tests don't import from app/ — they define their own inline functions.
# So the developer must fix the SERVICE LOGIC to match what the tests expect.

DEVELOPER_FIX_PROMPT = """You are the **Developer Agent** responsible for FIXING failing API code.

## Test Failure Output
{test_results}

## Current API Code (all files)
{api_code}

## Confirmed Specification
{spec}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

The tests above have FAILED. Analyse the failures carefully, identify the root cause,
and return a CORRECTED version of the full project.

IMPORTANT CONTEXT:
- Tests are SELF-CONTAINED — they define their own models and service functions inline.
- Tests do NOT import from the app/ package at all.
- Fix the corresponding logic in your service files so the test expectations would match.
- Every service function MUST accept a `db: Session` parameter.

Return a JSON object where each key is a filename and the value is the COMPLETE corrected file content.
Include ALL files, not just the ones you changed — the output replaces the entire codebase.
Return a single valid JSON object (no markdown fences).
"""


# ── PROMPT 7 of 8: Tester — Generate Self-Contained Tests ──
# This is the most carefully crafted prompt because test isolation is CRITICAL.
#
# WHY SELF-CONTAINED?
# - Tests can't import from app/ because the generated code might have import errors
# - Tests define their OWN SQLAlchemy models + service functions INLINE
# - This means tests can run even if the app code has syntax errors
# - Tests use only pytest + sqlalchemy (no FastAPI, no HTTP client)
#
# The "ABSOLUTE PROHIBITIONS" section exists because GPT-4o tends to
# generate import statements from app/ unless explicitly told not to.

TESTER_PROMPT = """You are the **Tester Agent** responsible for generating **self-contained local unit tests**.

## CRITICAL RULES — READ FIRST
1. Tests must be **100% self-contained**. Do NOT import anything from the `app/` package.
2. Do NOT import from `app.models`, `app.services`, `app.database`, or any generated code.
3. Do NOT use TestClient, httpx, FastAPI, or any HTTP-based testing.
4. The ONLY external dependencies allowed are: `pytest`, `sqlalchemy` (which are already installed).
5. Each test file defines its own SQLAlchemy models and service functions INLINE.

## API Code Files (for reference — extract the logic, don't import it)
{api_code}

## Confirmed Specification
{spec}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

## What to generate
Return a JSON object where keys are filenames (WITHOUT any `tests/` prefix) and values are file contents.

Required files:
- `__init__.py` — empty string.
- `conftest.py` — Shared fixtures ONLY using pytest and sqlalchemy (define models inline).
- `test_<entity>.py` — One file per entity with inline service functions + tests.

## ABSOLUTE PROHIBITIONS
- ❌ `from app.` anything
- ❌ `from fastapi` anything  
- ❌ `TestClient` or `httpx`

Return a single valid JSON object (no markdown fences).
"""


# ── PROMPT 8 of 8: DevOps — Generate Infrastructure Configs ──
# Input: list of all files in the project + spec + guidelines
# Output: JSON with Docker, Kubernetes, and CI/CD configuration files
#
# The DevOps Agent sees the FILE LIST (not the code itself) so it knows
# what files exist and can configure the Dockerfile COPY commands correctly.

DEVOPS_PROMPT = """You are the **DevOps Agent** responsible for CI/CD, containerisation, and orchestration configuration.

## Project Structure (filenames)
{file_list}

## Confirmed Specification
{spec}

## Company Guidelines (relevant sections retrieved via RAG)
{guidelines}

Generate all DevOps artefacts. Return a JSON object (keys = file paths, values = file content).

Required:
- `Dockerfile` — Multi-stage build, python:3.11-slim, non-root user, health-check.
- `docker-compose.yml` — Local dev: app + db service.
- `.github/workflows/ci-cd.yml` — GitHub Actions: lint, test, build image, push to GHCR.
- `k8s/deployment.yaml` — Deployment with resource limits, probes, env from ConfigMap/Secret.
- `k8s/service.yaml` — ClusterIP service.
- `k8s/configmap.yaml` — Non-sensitive config values.
- `k8s/hpa.yaml` — Horizontal Pod Autoscaler.
- `.dockerignore` — Standard Python ignores.

Return a single valid JSON object (no markdown fences).
"""

print("✅ All 8 prompt templates defined!")
print("   PROMPT 5: DEVELOPER_PROMPT — generate fresh code from spec")
print("   PROMPT 6: DEVELOPER_FIX_PROMPT — fix code after test failures")
print("   PROMPT 7: TESTER_PROMPT — generate self-contained pytest tests")
print("   PROMPT 8: DEVOPS_PROMPT — generate Docker/K8s/CI-CD configs")

✅ All 8 prompt templates defined!
   PROMPT 5: DEVELOPER_PROMPT — generate fresh code from spec
   PROMPT 6: DEVELOPER_FIX_PROMPT — fix code after test failures
   PROMPT 7: TESTER_PROMPT — generate self-contained pytest tests
   PROMPT 8: DEVOPS_PROMPT — generate Docker/K8s/CI-CD configs


---
# 🤖 SECTION 7: Agent State + Agent Classes
---
Now we define:
1. **AgentState** — The shared state (TypedDict) that flows through the entire LangGraph pipeline
2. **BAAgent** — Business Analyst (pre-pipeline, chat-based)
3. **ScrumMasterAgent** — Planning + Orchestration (the hub)
4. **DeveloperAgent** — Code generation + code fixing
5. **TesterAgent** — Test generation + pytest execution
6. **DevOpsAgent** — Docker/K8s/CI-CD + GitHub PR

In [252]:
# ============================================================================
# STEP 7.1: AgentState — The Shared Whiteboard  (agents/state.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# AgentState is a TypedDict that defines ALL the data flowing through the
# LangGraph pipeline. Think of it as a "shared whiteboard" that all 5 agents
# can read from and write to. Each agent:
#   - READS the fields it needs (e.g., developer reads "spec" and "planning_output")
#   - WRITES its results back (e.g., developer writes "api_code")
#
# WHY TypedDict?
# - LangGraph requires a typed state definition so it knows what data exists
# - TypedDict gives us type hints but no runtime enforcement (like a dict)
# - The Annotated[..., add_messages] on "messages" tells LangGraph to APPEND
#   new messages instead of replacing them (accumulator pattern)
#
# FLOW OF DATA THROUGH THE PIPELINE:
# ───────────────────────────────────
#   START     → spec, guidelines_context, db_schema are set
#   Planning  → writes planning_output, sets current_phase = "planning_complete"
#   Orchestr. → reads current_phase, writes next_agent
#   Developer → reads spec + planning, writes api_code, sets current_phase = "development_complete"
#   Orchestr. → reads current_phase, writes next_agent = "tester"
#   Tester    → reads api_code, writes test_code + test_results + test_passed
#   Orchestr. → reads test_passed, writes next_agent = "devops" or "developer"
#   DevOps    → reads everything, writes devops_config + pr_result
#   Orchestr. → reads current_phase = "devops_complete", writes next_agent = "end"
#
# The build_log accumulates entries from EVERY agent — it's the chronological
# record of everything that happened during the pipeline run.


class AgentState(TypedDict):
    """Shared state that flows through the multi-agent LangGraph pipeline."""

    # LangGraph message accumulator — append-only (add_messages reducer)
    messages: Annotated[Sequence[BaseMessage], add_messages]

    # ── Inputs (set once at pipeline start, read by all agents) ───
    spec: dict                 # The confirmed API specification (from BA Agent)
    guidelines_context: str    # Legacy/fallback guidelines string
    db_schema: str             # Database schema + sample data (for developer context)
    project_name: str          # Project identifier (used for git branch name, etc.)

    # ── Phase Outputs (each agent writes its results here) ────────
    planning_output: dict      # Scrum Master → {jira_tickets, confluence_doc, flowchart, readme}
    api_code: dict             # Developer → {filename: source_code}
    test_code: dict            # Tester → {filename: test_source_code}
    test_results: str          # Tester → pytest stdout/stderr output
    test_passed: bool          # Tester → True if all tests passed
    devops_config: dict        # DevOps → {filename: config_content}
    pr_result: dict            # DevOps → {success, pr_url, pr_number}

    # ── Orchestration Control (read/written by orchestrator) ──────
    current_phase: str         # Where we are: "planning_complete", "development_complete", etc.
    next_agent: str            # Orchestrator's routing decision: "developer"/"tester"/"devops"/"end"
    retry_count: int           # How many dev↔test fix loops have occurred so far
    max_retries: int           # Cap on fix attempts (default 3, set at pipeline start)
    build_log: list[str]       # Chronological trace of ALL agent actions (append-only)


print("✅ AgentState defined — the shared state for the entire pipeline")
print("   14 fields total: 4 inputs, 7 outputs, 3 orchestration controls")

✅ AgentState defined — the shared state for the entire pipeline
   14 fields total: 4 inputs, 7 outputs, 3 orchestration controls


In [253]:
# ============================================================================
# STEP 7.2: BA Agent Class  (agents/ba_agent.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# The BA Agent runs OUTSIDE the LangGraph pipeline — it's used in the
# Streamlit chat interface for interactive spec gathering. Unlike other
# agents which are graph nodes, the BA is a standalone class.
#
# TWO CAPABILITIES:
# 1. conduct_conversation() — Multi-turn chat with the user
#    - Called on every user message in the Streamlit chat
#    - Returns the assistant's response
#    - Uses BA_CONVERSATION_SYSTEM prompt with RAG guidelines + DB schema
#
# 2. extract_spec() — Extract structured JSON spec from conversation
#    - Called when user confirms the spec
#    - Concatenates ALL messages into one text block
#    - Uses BA_SPEC_EXTRACTION_PROMPT to parse into JSON
#    - The output JSON becomes input to the entire pipeline
#
# FLOW:
# ─────
#   User types request → conduct_conversation() → response displayed
#   User types more → conduct_conversation() → response displayed
#   ... (multi-turn, 3-4 questions at a time)
#   User confirms → extract_spec() → JSON spec → pipeline starts
#
# WHY EACH AGENT GETS ITS OWN LLM INSTANCE:
# - Different temperatures (BA uses 0.3 for balanced, Developer uses 0.2 for precise)
# - Independent token tracking
# - Easy to swap models per agent in the future


class BAAgent:
    """
    Business Analyst Agent — handles interactive spec-driven conversation
    and structured specification extraction.
    """
    def __init__(self, model: str, api_key: str, temperature: float = 0.3):
        # Each agent creates its own ChatOpenAI instance — no sharing
        self.llm = ChatOpenAI(model=model, api_key=api_key, temperature=temperature)

    def conduct_conversation(
        self, user_message: str, chat_history: list[dict],
        guidelines_context: str, db_schema: str = "",
    ) -> str:
        """
        Return the next assistant message in the spec conversation.
        
        How it works step by step:
        1. Build system prompt → inject RAG guidelines + DB schema into placeholders
        2. Replay full conversation history as LangChain message objects
        3. Append the new user message
        4. Call GPT-4o with the full message list
        5. Return the assistant's text response
        """
        print(f"\n🧑‍💼 [BA Agent] Processing user message ({len(user_message)} chars)")
        
        # Step 1: Build the system message with dynamic context
        # The {guidelines_context} and {db_schema} placeholders get filled here
        system = BA_CONVERSATION_SYSTEM.format(
            guidelines_context=guidelines_context,
            db_schema=db_schema,
        )
        messages = [SystemMessage(content=system)]

        # Step 2: Replay conversation history as proper message types
        # LangChain needs HumanMessage/AIMessage objects, not raw dicts
        for msg in chat_history:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            else:
                messages.append(AIMessage(content=msg["content"]))

        # Step 3: Add the new user message
        messages.append(HumanMessage(content=user_message))

        # Step 4: Call the LLM with the full message sequence
        response = self.llm.invoke(messages)
        print(f"✅ [BA Agent] Response generated ({len(response.content)} chars)")
        return response.content

    def extract_spec(self, chat_history: list[dict], guidelines_context: str) -> dict:
        """
        Parse the entire conversation into a structured JSON specification.
        
        Called when the user clicks "Confirm Specification" in the UI.
        Takes ALL messages from the chat and produces a single JSON object
        that becomes the input to the entire downstream pipeline.
        """
        print("\n🔍 [BA Agent] Extracting structured specification from conversation...")
        
        # Concatenate all messages into one text block for the extraction prompt
        conversation_text = "\n".join(
            f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content']}"
            for m in chat_history
        )
        
        # Fill the extraction prompt template with conversation + guidelines
        prompt = BA_SPEC_EXTRACTION_PROMPT.format(
            conversation=conversation_text,
            guidelines=guidelines_context,
        )
        
        # Call the LLM — it returns a JSON string
        response = self.llm.invoke([HumanMessage(content=prompt)])
        
        # Parse the JSON response using our 3-tier fallback parser
        # (handles markdown fences, truncated JSON, etc.)
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [BA Agent] Spec extracted — {len(result)} top-level keys: {list(result.keys())}")
        return result


print("✅ BAAgent class defined — pre-pipeline chat + spec extraction")

✅ BAAgent class defined — pre-pipeline chat + spec extraction


In [254]:
# ============================================================================
# STEP 7.3: Scrum Master Agent  (agents/scrum_master.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# The Scrum Master is the MOST IMPORTANT agent in the pipeline because it
# controls the FLOW. It has TWO distinct roles:
#
#   ROLE 1 — PLANNER (planning_node):
#     Input: confirmed spec JSON
#     Output: {jira_tickets, confluence_doc, flowchart, readme}
#     Called ONCE at the start of the pipeline
#
#   ROLE 2 — ORCHESTRATOR (orchestrator_node):
#     Input: full AgentState (sees everything)
#     Output: next_agent = "developer" / "tester" / "devops" / "end"
#     Called AFTER EVERY OTHER AGENT — this is the hub of hub-and-spoke
#
# WHY HUB-AND-SPOKE?
# ──────────────────
#   Traditional: Planning → Dev → Test → DevOps → END (linear, no feedback)
#   Hub-and-spoke: Every agent → Orchestrator → decides next agent
#
#   The orchestrator enables the SELF-HEALING LOOP:
#     Developer → Tester → FAIL → Developer (fix) → Tester → PASS → DevOps
#   Without the orchestrator, there's no way to route back for fixes.
#
# NODE FUNCTIONS:
# ──────────────
# Each LangGraph node is a FUNCTION (not a class method). The function:
#   1. Creates a fresh agent instance
#   2. Calls the appropriate method
#   3. Returns a dict that gets MERGED into AgentState


class ScrumMasterAgent:
    """
    Scrum Master Agent — Planning + Orchestration.
    
    This is the most important agent because it controls the FLOW.
    After every agent finishes, the orchestrator decides what happens next.
    """
    def __init__(self, model: str, api_key: str, temperature: float = 0.3):
        self.llm = ChatOpenAI(model=model, api_key=api_key, temperature=temperature)

    def generate_planning(self, spec: dict, guidelines: str) -> dict:
        """
        Generate planning artefacts from the confirmed spec.
        
        Returns: {jira_tickets, confluence_doc, flowchart, readme}
        These get saved as .md files in the output directory by DevOps.
        """
        print("\n📋 [Scrum Master] Generating planning artefacts...")
        prompt = PLANNING_PROMPT.format(
            spec=json.dumps(spec, indent=2),
            guidelines=guidelines,
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [Scrum Master] Planning complete — artefacts: {list(result.keys())}")
        return result

    def decide_next(self, state: AgentState) -> str:
        """
        Use LLM to decide which agent should run next.
        
        This is the ORCHESTRATOR brain — it sees the full pipeline state
        and decides: "developer", "tester", "devops", or "end"
        
        The LLM sees the current_phase, test_passed, retry_count, etc.
        and applies the decision rules from ORCHESTRATOR_PROMPT.
        """
        print(f"\n🧠 [Orchestrator] Evaluating state — phase={state.get('current_phase')}, "
              f"test_passed={state.get('test_passed')}, retries={state.get('retry_count', 0)}")
        
        # Show last 5 build log entries to the LLM for decision context
        recent_log = state.get("build_log", [])[-5:]
        
        # Fill the orchestrator prompt with current pipeline state
        prompt = ORCHESTRATOR_PROMPT.format(
            current_phase=state.get("current_phase", "unknown"),
            has_api_code=bool(state.get("api_code")),
            has_test_code=bool(state.get("test_code")),
            test_passed=state.get("test_passed", False),
            has_devops=bool(state.get("devops_config")),
            retry_count=state.get("retry_count", 0),
            max_retries=state.get("max_retries", 3),
            recent_log="\n".join(f"  - {l}" for l in recent_log) if recent_log else "  (none)",
        )
        
        # The LLM should return a single word
        response = self.llm.invoke([HumanMessage(content=prompt)])
        decision = response.content.strip().lower()
        
        # Normalise to valid choices (in case LLM adds extra text)
        for valid in ("developer", "tester", "devops", "end"):
            if valid in decision:
                print(f"➡️  [Orchestrator] Decision: {valid.upper()}")
                return valid
        print("➡️  [Orchestrator] Decision: END (default fallback)")
        return "end"


# ── LangGraph Node Functions for Scrum Master ──
# These are standalone functions (not class methods) because LangGraph
# nodes must be callables that take state and return a dict.

def planning_node(state: AgentState) -> dict:
    """
    LangGraph node — Scrum Master produces planning artefacts.
    
    Called ONCE at pipeline start (the entry point of the graph).
    Uses the main guideline retriever (not agent-specific) because
    planning needs broad context about the whole project.
    """
    agent = ScrumMasterAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)
    
    # Retrieve guidelines relevant to planning
    retriever = get_retriever()
    guidelines = retriever.retrieve(
        f"project planning API design architecture {state['spec'].get('project_name', '')}", top_k=8,
    )
    
    planning = agent.generate_planning(state["spec"], guidelines)
    
    # Return dict gets MERGED into AgentState by LangGraph
    return {
        "planning_output": planning,
        "current_phase": "planning_complete",  # Orchestrator reads this to decide "developer"
        "build_log": state.get("build_log", [])
        + ["[Scrum Master] Planning phase completed."],
    }


def orchestrator_node(state: AgentState) -> dict:
    """
    LangGraph node — Scrum Master decides which agent runs next.
    
    Called AFTER EVERY OTHER AGENT completes. The conditional edge
    function (_route_from_orchestrator) reads state["next_agent"]
    to determine the actual graph routing.
    """
    agent = ScrumMasterAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)
    decision = agent.decide_next(state)
    return {
        "next_agent": decision,  # _route_from_orchestrator reads this
        "build_log": state.get("build_log", [])
        + [f"[Scrum Master] Orchestrator decided next agent → {decision}"],
    }


print("✅ ScrumMasterAgent + planning_node + orchestrator_node defined!")
print("   planning_node → entry point, generates Jira/Confluence/flowchart/README")
print("   orchestrator_node → the hub, called after every agent")

✅ ScrumMasterAgent + planning_node + orchestrator_node defined!
   planning_node → entry point, generates Jira/Confluence/flowchart/README
   orchestrator_node → the hub, called after every agent


In [255]:
# ============================================================================
# STEP 7.4: Developer Agent  (agents/developer.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# The Developer Agent generates FastAPI source code from the spec.
# It has TWO MODES:
#
#   MODE 1 — GENERATE (first run):
#     Orchestrator routes here after planning_complete
#     Input: spec + planning_output + guidelines + db_schema
#     Output: {filename: source_code} for the entire API project
#     Uses: DEVELOPER_PROMPT
#
#   MODE 2 — FIX (self-healing loop):
#     Orchestrator routes here after tests FAIL
#     Input: current code + test failure output + spec + guidelines
#     Output: CORRECTED {filename: source_code} (replaces entire codebase)
#     Uses: DEVELOPER_FIX_PROMPT
#
# HOW THE NODE DECIDES WHICH MODE:
# ─────────────────────────────────
#   if api_code already exists AND test_results exist AND test_passed is False:
#       → FIX mode (repair the code)
#   else:
#       → GENERATE mode (create fresh code)
#
# RAG:
# ────
# The developer uses get_developer_retriever() which loads from
# developer_guidelines.txt — this file contains coding standards,
# response format rules, naming conventions, etc.
# The DB schema is ALSO appended to the guidelines so the developer
# knows the real table structure.


class DeveloperAgent:
    """
    Developer Agent — generates or fixes API code.
    
    When tests fail, the orchestrator routes back to the developer.
    The developer reads the test output and regenerates corrected code.
    This is the "self-healing" loop.
    """
    def __init__(self, model: str, api_key: str, temperature: float = 0.2):
        # Lower temperature (0.2) for more precise, deterministic code generation
        self.llm = ChatOpenAI(model=model, api_key=api_key, temperature=temperature)

    def generate_code(self, spec: dict, planning: dict, guidelines: str) -> dict:
        """
        GENERATE mode: Create fresh API code from spec.
        Returns {filename: source_code} for the complete FastAPI project.
        """
        print("\n💻 [Developer] Generating API code from spec...")
        prompt = DEVELOPER_PROMPT.format(
            spec=json.dumps(spec, indent=2),
            planning=json.dumps(planning, indent=2),
            guidelines=guidelines,
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [Developer] Code generated — {len(result)} files: {list(result.keys())}")
        return result

    def fix_code(self, api_code: dict, test_results: str, spec: dict, guidelines: str) -> dict:
        """
        FIX mode: Analyse test failures and return corrected code.
        The test_results contain pytest output showing exactly what failed.
        """
        print("\n🔧 [Developer] Fixing code based on test failures...")
        prompt = DEVELOPER_FIX_PROMPT.format(
            test_results=test_results,
            api_code=json.dumps(api_code, indent=2),
            spec=json.dumps(spec, indent=2),
            guidelines=guidelines,
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [Developer] Code fix complete — {len(result)} files regenerated")
        return result


# ── LangGraph Node Function for Developer ──

def development_node(state: AgentState) -> dict:
    """
    LangGraph node — Developer Agent.
    
    This node is called when the orchestrator decides next_agent = "developer".
    It auto-detects whether to GENERATE or FIX based on pipeline state.
    
    After finishing, the orchestrator will route to "tester" to validate
    the generated/fixed code.
    """
    agent = DeveloperAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)

    # Retrieve developer-specific guidelines via RAG
    # Uses developer_guidelines.txt (not the generic guideline.txt)
    retriever = get_developer_retriever()
    guidelines = retriever.retrieve(
        f"FastAPI code generation models schemas routes response format database SQLAlchemy "
        f"{state['spec'].get('project_name', '')}",
        top_k=8,
    )
    print(f"📄 [RAG → Developer Guidelines] Retrieved {len(guidelines)} chars")

    # Append DB schema so developer knows the real tables/columns
    db_schema = state.get("db_schema", "")
    if db_schema:
        guidelines = guidelines + "\n\n" + db_schema

    retry_count = state.get("retry_count", 0)
    existing_code = state.get("api_code", {})
    test_results = state.get("test_results", "")

    # AUTO-DETECT: GENERATE vs FIX mode
    if existing_code and test_results and not state.get("test_passed", True):
        # ── FIX mode: tests failed, repair the code ──
        code = agent.fix_code(existing_code, test_results, state["spec"], guidelines)
        log_msg = f"[Developer Agent] Code fix attempt #{retry_count + 1} — {len(code)} files regenerated."
    else:
        # ── GENERATE mode: first run, create fresh code ──
        code = agent.generate_code(state["spec"], state.get("planning_output", {}), guidelines)
        log_msg = f"[Developer Agent] Initial code generation — {len(code)} files created."

    return {
        "api_code": code,
        "current_phase": "development_complete",  # Orchestrator reads this → routes to "tester"
        "build_log": state.get("build_log", []) + [log_msg],
    }


print("✅ DeveloperAgent + development_node defined!")
print("   Two modes: GENERATE (fresh code) and FIX (self-healing after test failure)")

✅ DeveloperAgent + development_node defined!
   Two modes: GENERATE (fresh code) and FIX (self-healing after test failure)


In [256]:
# ============================================================================
# STEP 7.5: Tester Agent  (agents/tester.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# The Tester Agent does TWO things in sequence:
#   1. GENERATE tests — GPT-4o creates pytest test files from the API code
#   2. EXECUTE tests — runs pytest via subprocess and captures the output
#
# KEY DESIGN DECISION: Self-Contained Tests
# ──────────────────────────────────────────
# Tests do NOT import from app/. Instead, they define their own:
#   - SQLAlchemy models (inline, mirroring the app models)
#   - Service functions (inline, mirroring the app services)
#   - Fixtures (conftest.py with in-memory SQLite)
#
# WHY? Because the generated app code might have import errors, missing
# dependencies, or structural issues. Self-contained tests can still run
# and validate the LOGIC even if the app package is broken.
#
# THE 3-STEP PROCESS:
# ───────────────────
#   Step 1: generate_tests() — LLM creates test files as {filename: content}
#   Step 2: save_artifacts() — writes api/ and tests/ to disk
#   Step 3: run_tests() — executes `python -m pytest tests/ -v --tb=short -x`
#
# WHAT HAPPENS AFTER:
# ───────────────────
#   Tests PASS → orchestrator routes to DevOps (pipeline continues)
#   Tests FAIL → retry_count += 1 → orchestrator routes to Developer (fix loop)
#   Tests FAIL + max retries → orchestrator routes to DevOps (best effort)


class TesterAgent:
    """
    Tester Agent — generates and executes unit tests.
    
    The tests are LOCAL function-level tests (no API server needed).
    If tests fail, the orchestrator routes back to the developer for fixes.
    """
    def __init__(self, model: str, api_key: str, temperature: float = 0.2):
        self.llm = ChatOpenAI(model=model, api_key=api_key, temperature=temperature)

    def generate_tests(self, api_code: dict, spec: dict, guidelines: str) -> dict:
        """
        Use GPT-4o to generate self-contained test files.
        
        The LLM sees the api_code as reference but generates tests that
        DON'T import from it — they define everything inline.
        
        Returns: {filename: test_source_code} (no tests/ prefix)
        """
        print("\n🧪 [Tester] Generating local unit tests (function-level, no API server)...")
        prompt = TESTER_PROMPT.format(
            api_code=json.dumps(api_code, indent=2),
            spec=json.dumps(spec, indent=2),
            guidelines=guidelines,
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [Tester] Tests generated — {len(result)} test files: {list(result.keys())}")
        return result

    @staticmethod
    def run_tests(output_dir: str) -> tuple:
        """
        Execute the generated unit tests via pytest subprocess.
        
        HOW IT WORKS:
        1. Find the tests/ directory inside output_dir
        2. Set PYTHONPATH so pytest can find conftest.py and test files
        3. Run: python -m pytest tests/ -v --tb=short -x
           -v: verbose (show each test name + pass/fail)
           --tb=short: short tracebacks (enough to debug, not too noisy)
           -x: stop on first failure (fail fast)
        4. Return (output_text, passed_bool)
        
        The output_text goes into state["test_results"] — if tests fail,
        the Developer Agent reads this to understand what went wrong.
        """
        import subprocess
        tests_dir = os.path.abspath(os.path.join(output_dir, "tests"))
        api_dir = os.path.abspath(os.path.join(output_dir, "api"))

        if not os.path.isdir(tests_dir):
            return "No tests directory found — skipping execution.", False

        run_cwd = os.path.abspath(output_dir)

        # Build PYTHONPATH so imports resolve correctly
        # tests_dir → conftest.py imports work
        # api_dir   → backward compatibility
        # run_cwd   → top-level imports
        env = os.environ.copy()
        existing = env.get("PYTHONPATH", "")
        paths = os.pathsep.join(filter(None, [tests_dir, api_dir, run_cwd, existing]))
        env["PYTHONPATH"] = paths

        try:
            print("🏃 [Tester] Running local unit tests (no server required)...")
            result = subprocess.run(
                ["python", "-m", "pytest", tests_dir, "-v", "--tb=short", "-x"],
                capture_output=True,  # Capture stdout + stderr
                text=True,            # Return as string (not bytes)
                cwd=run_cwd,
                env=env,
                timeout=60,           # 60-second timeout to prevent hangs
            )
            output = (result.stdout + "\n" + result.stderr).strip()
            passed = result.returncode == 0  # pytest returns 0 on success
            if not output:
                output = "Tests executed — no output captured."
            return output, passed
        except FileNotFoundError:
            return "pytest not found — install it to run tests.", False
        except subprocess.TimeoutExpired:
            return "Test execution timed out after 60 seconds.", False
        except Exception as e:
            return f"Test execution error: {e}", False


# ── LangGraph Node Function for Tester ──

def testing_node(state: AgentState) -> dict:
    """
    LangGraph node — Tester Agent.
    
    Full sequence:
      1. Retrieve tester-specific guidelines via RAG
      2. Generate test files using GPT-4o
      3. Clean up filenames (strip "tests/" prefix to avoid nesting)
      4. Save api + test files to disk (output_api/)
      5. Execute pytest via subprocess
      6. Report pass/fail + increment retry_count if failed
    
    The orchestrator reads test_passed to decide:
      True  → route to devops
      False → route to developer (fix loop)
    """
    agent = TesterAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)
    output_dir = settings.OUTPUT_DIR

    # Get tester-specific RAG guidelines (from tester_guidelines.txt)
    retriever = get_tester_retriever()
    guidelines = retriever.retrieve(
        f"pytest test cases API validation error handling fixtures conftest "
        f"{state['spec'].get('project_name', '')}",
        top_k=8,
    )
    print(f"📄 [RAG → Tester Guidelines] Retrieved {len(guidelines)} chars")

    # Step 1: Generate test code via LLM
    test_code = agent.generate_tests(state.get("api_code", {}), state["spec"], guidelines)

    # Step 2: Clean filenames — strip "tests/" prefix
    # The LLM sometimes returns keys like "tests/test_book.py"
    # but we save into a tests/ directory, so we'd get tests/tests/test_book.py
    cleaned_tests = {}
    for k, v in test_code.items():
        clean_key = k.replace("tests/", "").replace("tests\\\\", "") if k.startswith("tests") else k
        cleaned_tests[clean_key] = v
    test_code = cleaned_tests

    # Step 3: Save api code + tests to disk so pytest can find them
    save_artifacts(output_dir, {"api": state.get("api_code", {}), "tests": test_code})

    # Step 4: Execute tests via pytest subprocess
    print("\n🏃 [Tester] Executing tests via pytest...")
    test_output, passed = TesterAgent.run_tests(output_dir)

    # Step 5: Report results and update retry count
    status = "PASSED ✅" if passed else "FAILED ❌"
    print(f"{'✅' if passed else '❌'} [Tester] Test execution: {status}")
    retry_count = state.get("retry_count", 0)
    new_retry = retry_count if passed else retry_count + 1  # Only increment on failure

    return {
        "test_code": test_code,
        "test_results": test_output,     # Developer reads this in FIX mode
        "test_passed": passed,           # Orchestrator reads this for routing
        "retry_count": new_retry,
        "current_phase": "testing_complete",
        "build_log": state.get("build_log", [])
        + [
            f"[Tester Agent] {len(test_code)} test files generated.",
            f"[Tester Agent] Test execution: {status}",
            f"[Tester Agent] Retry count: {new_retry}/{state.get('max_retries', 3)}",
        ],
    }


print("✅ TesterAgent + testing_node defined!")
print("   Generates self-contained tests → saves to disk → runs pytest → reports pass/fail")

✅ TesterAgent + testing_node defined!
   Generates self-contained tests → saves to disk → runs pytest → reports pass/fail


In [257]:
# ============================================================================
# STEP 7.6: DevOps Agent  (agents/devops.py)
# ============================================================================
#
# WHAT THIS IS:
# -------------
# The DevOps Agent is the FINAL agent in the pipeline. When it finishes,
# the orchestrator routes to END and the pipeline is complete.
#
# THREE RESPONSIBILITIES:
# 1. Generate infrastructure configs (Docker, K8s, CI/CD)
# 2. Save ALL project artefacts to disk (api, tests, devops, planning)
# 3. Initialize git repo → create branch → push → create GitHub PR
#
# WHERE IN THE PIPELINE:
# ──────────────────────
#   Planning → Dev → Tester → (fix loop) → DevOps → END
#                                              ↑
#                                         You are here
#
# BY THIS POINT, the state contains:
#   - spec (from BA)
#   - planning_output (from Scrum Master)
#   - api_code (from Developer — possibly fixed)
#   - test_code + test_results (from Tester)
# So DevOps can save EVERYTHING to disk.
#
# THE PR STEP:
# ────────────
# If GITHUB_TOKEN, GITHUB_REPO_OWNER, and GITHUB_REPO_NAME are configured,
# the DevOps Agent will:
#   1. git init in the output directory
#   2. Create a feature branch (e.g., "feature/library-api-stub")
#   3. Commit all files
#   4. Push to GitHub
#   5. Create a Pull Request via GitHub REST API
# If credentials aren't set, the PR step is silently skipped.


class DevOpsAgent:
    """
    DevOps Agent — generates infrastructure configs and creates PRs.
    
    This is the FINAL agent in the pipeline. After DevOps completes,
    the orchestrator routes to END (pipeline done).
    """
    def __init__(self, model: str, api_key: str, temperature: float = 0.2):
        self.llm = ChatOpenAI(model=model, api_key=api_key, temperature=temperature)

    def generate_configs(self, file_list: list, spec: dict, guidelines: str) -> dict:
        """
        Generate infrastructure configuration files.
        
        The LLM sees the LIST of files (not their content) so it knows
        what the project structure looks like. Returns files like
        Dockerfile, docker-compose.yml, k8s/*.yaml, CI/CD workflow.
        """
        print("\n🐳 [DevOps] Generating Docker, K8s, and CI/CD configs...")
        prompt = DEVOPS_PROMPT.format(
            file_list="\n".join(file_list),
            spec=json.dumps(spec, indent=2),
            guidelines=guidelines,
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        result = parse_llm_json(response.content, self.llm)
        print(f"✅ [DevOps] Configs generated — {len(result)} files: {list(result.keys())}")
        return result


# ── LangGraph Node Function for DevOps ──

def devops_node(state: AgentState) -> dict:
    """
    LangGraph node — DevOps Agent (FINAL node in the pipeline).
    
    Full sequence:
      1. Retrieve DevOps-specific guidelines via RAG
      2. Generate Docker/K8s/CI-CD configs via LLM
      3. Collect ALL artefacts (api + tests + devops + planning)
      4. Save everything to disk using save_artifacts()
      5. If GitHub credentials are set:
         a. git init → create branch → commit
         b. Push to GitHub → create Pull Request
      6. Return devops_config + pr_result to state
    
    After this node, the orchestrator routes to END.
    """
    agent = DevOpsAgent(model=settings.OPENAI_MODEL, api_key=settings.OPENAI_API_KEY)

    # Retrieve DevOps-specific RAG guidelines (from devops_guidelines.txt)
    retriever = get_devops_retriever()
    guidelines = retriever.retrieve(
        "Docker Kubernetes CI/CD GitHub Actions deployment pipeline containerisation PR",
        top_k=8,
    )
    print(f"📄 [RAG → DevOps Guidelines] Retrieved {len(guidelines)} chars")

    # Build file list for the DevOps prompt (api files + test files)
    file_list = list(state.get("api_code", {}).keys()) + list(state.get("test_code", {}).keys())

    # Step 1: Generate configs
    configs = agent.generate_configs(file_list, state["spec"], guidelines)

    # Step 2: Collect ALL project artefacts for disk save
    output_dir = settings.OUTPUT_DIR
    artefacts = {}
    if state.get("api_code"):
        artefacts["api"] = state["api_code"]       # Developer's code
    if state.get("test_code"):
        artefacts["tests"] = state["test_code"]     # Tester's tests
    artefacts["devops"] = configs                    # DevOps configs

    # Save planning docs as markdown files
    planning = state.get("planning_output", {})
    plan_files = {}
    for key in ("jira_tickets", "confluence_doc", "flowchart", "readme"):
        if key in planning:
            plan_files[f"{key}.md"] = planning[key]
    if plan_files:
        artefacts["planning"] = plan_files

    # Step 3: Save everything to disk
    save_artifacts(output_dir, artefacts)
    print(f"💾 [DevOps] Saved all artefacts to {output_dir}/")

    # Step 4: Git + GitHub PR (optional — requires credentials)
    pr_result = {}
    gh_token = settings.GITHUB_TOKEN
    gh_owner = settings.GITHUB_REPO_OWNER
    gh_repo = settings.GITHUB_REPO_NAME

    if gh_token and gh_owner and gh_repo:
        project_name = state.get("project_name", "api-project")
        project_dir = os.path.abspath(output_dir)
        branch = f"feature/{project_name}-stub"
        pr_title = f"feat: {project_name} — AI-generated API stub"

        print(f"\n🔀 [DevOps] Initialising git repo and creating branch: {branch}")
        init_local_repo(project_dir)
        create_branch_and_commit(project_dir, branch, pr_title)

        print(f"📤 [DevOps] Pushing to GitHub and creating PR → {gh_owner}/{gh_repo}")
        # PR body includes the spec JSON + build log for traceability
        spec_json = json.dumps(state["spec"], indent=2)
        build_log = "\n".join(f"- {e}" for e in state.get("build_log", []))
        pr_body = (
            f"## Auto-generated by AI-SDLC Accelerator\n\n"
            f"**Spec:**\n```json\n{spec_json}\n```\n\n"
            f"**Build Log:**\n{build_log}"
        )
        pr_result = push_and_create_pr(
            project_dir, branch, pr_title, pr_body,
            gh_token, gh_owner, gh_repo,
        )
        if pr_result.get("success"):
            print(f"✅ [DevOps] PR created: {pr_result['pr_url']}")
        else:
            print(f"⚠️  [DevOps] PR failed: {pr_result.get('error', 'Unknown')}")
    else:
        print("⚠️  [DevOps] GitHub credentials not configured — skipping PR")

    return {
        "devops_config": configs,
        "pr_result": pr_result,
        "current_phase": "devops_complete",  # Orchestrator reads this → routes to "end"
        "build_log": state.get("build_log", [])
        + [
            f"[DevOps Agent] {len(configs)} configuration files generated (Docker, K8s, CI/CD).",
            f"[DevOps Agent] Artefacts saved to disk.",
            f"[DevOps Agent] PR: {pr_result.get('pr_url', 'skipped')}",
        ],
    }


print("✅ DevOpsAgent + devops_node defined!")
print("   Final pipeline node: generates configs → saves all artefacts → creates PR")

✅ DevOpsAgent + devops_node defined!
   Final pipeline node: generates configs → saves all artefacts → creates PR


---
## 📌 Section 8: LangGraph Pipeline — The Brain of the System

This is where everything comes together. We build a **StateGraph** that wires all 5 agents into a **hub-and-spoke** architecture:

```
Planning → Orchestrator ← → Developer
                        ← → Tester
                        ← → DevOps
                        → END
```

The **Orchestrator** (Scrum Master) is the central hub. After each agent finishes, control returns to the orchestrator, which decides the next step based on the full pipeline state.

In [258]:
# ============================================================================
# STEP 8.1: Build the LangGraph Pipeline  (agents/graph.py)
# ============================================================================
#
# THIS IS THE CORE OF THE ENTIRE SYSTEM.
#
# Everything we've defined so far (agents, prompts, RAG, utilities) gets
# wired together into a single LangGraph StateGraph here.
#
# ARCHITECTURE — Hub-and-Spoke:
# ─────────────────────────────
#   ┌──────────────┐
#   │   Planning   │  ← Entry point (Scrum Master generates Jira/Confluence)
#   └──────┬───────┘
#          ▼
#   ┌──────────────────┐
#   │  ORCHESTRATOR    │◄───── EVERY agent reports back to this hub
#   │ (Scrum Master)   │
#   └──┬───┬───┬───┬──┘
#      │   │   │   │
#      ▼   ▼   ▼   ▼
#    DEV TEST DEVOPS END
#
# HOW LANGGRAPH WORKS:
# ────────────────────
# 1. StateGraph(AgentState) — creates a graph where state flows through nodes
# 2. add_node("name", function) — registers a node (agent function)
# 3. add_edge("A", "B") — fixed edge: after A, always go to B
# 4. add_conditional_edges("A", func, map) — dynamic: func returns a key,
#    map says which key goes to which node
# 5. set_entry_point("name") — where the graph starts
# 6. compile(checkpointer) — produces a runnable graph
#
# THE CONDITIONAL EDGE (the magic):
# ──────────────────────────────────
# After the orchestrator runs, _route_from_orchestrator() reads
# state["next_agent"] and routes to the appropriate node.
# This is what makes the flow DYNAMIC — the LLM decides where to go!
#
# MEMORY SAVER:
# ─────────────
# MemorySaver is a checkpointer that stores the state after each node.
# It allows the graph to resume from any checkpoint if needed.
# Without it, the graph would lose state if interrupted.


from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


def _route_from_orchestrator(state: AgentState) -> str:
    """
    Conditional edge function — reads the orchestrator's decision.
    
    The orchestrator writes state["next_agent"] = "developer"/"tester"/"devops"/"end"
    This function reads that value and returns it as a routing key.
    LangGraph then follows the edge map to the corresponding node.
    
    This is what makes the pipeline DYNAMIC — the LLM decides the flow!
    """
    next_agent = state.get("next_agent", "end")
    if next_agent in ("developer", "tester", "devops"):
        return next_agent
    return "end"  # Default to end if unknown


def build_pipeline():
    """
    Construct and compile the multi-agent LangGraph pipeline.
    
    Returns a compiled StateGraph ready to be invoked with initial state.
    
    The graph has:
      - 5 nodes: planning, orchestrator, developer, tester, devops
      - 1 entry point: planning
      - 3 fixed edges: planning→orchestrator, dev→orch, test→orch, devops→orch
      - 1 conditional edge: orchestrator → {developer, tester, devops, END}
    """
    workflow = StateGraph(AgentState)

    # ── Step 1: Register all agent nodes ──────────────────────────
    # Each node is a function that takes AgentState and returns a dict
    workflow.add_node("planning", planning_node)          # Scrum Master plans
    workflow.add_node("orchestrator", orchestrator_node)  # Scrum Master routes
    workflow.add_node("developer", development_node)      # Developer generates/fixes code
    workflow.add_node("tester", testing_node)              # Tester generates + runs tests
    workflow.add_node("devops", devops_node)                # DevOps generates configs + PR

    # ── Step 2: Wire the flow ─────────────────────────────────────

    # The pipeline ALWAYS starts with planning
    workflow.set_entry_point("planning")

    # After planning → go to orchestrator (who will route to developer)
    workflow.add_edge("planning", "orchestrator")

    # Orchestrator dynamically routes based on LLM decision
    # _route_from_orchestrator reads state["next_agent"] and returns a key
    # The dict maps each key to the corresponding node
    workflow.add_conditional_edges(
        "orchestrator",               # Source node
        _route_from_orchestrator,      # Function that returns a routing key
        {                              # Map of routing keys → destination nodes
            "developer": "developer",
            "tester": "tester",
            "devops": "devops",
            "end": END,                # END is a special LangGraph constant
        },
    )

    # ALL agents report back to orchestrator (hub-and-spoke pattern)
    # This creates the feedback loop: dev → orch → test → orch → dev (if fail)
    workflow.add_edge("developer", "orchestrator")
    workflow.add_edge("tester", "orchestrator")
    workflow.add_edge("devops", "orchestrator")

    # ── Step 3: Compile with memory checkpointer ─────────────────
    # MemorySaver stores state snapshots after each node for resumability
    memory = MemorySaver()
    return workflow.compile(checkpointer=memory)


print("✅ build_pipeline() defined!")
print("   5 nodes: planning, orchestrator, developer, tester, devops")
print("   Hub-and-spoke: all agents route back to orchestrator")
print("   Conditional edge: orchestrator decides next agent dynamically")
print("   Self-healing loop: dev → test → fail → dev → test → pass → devops")

✅ build_pipeline() defined!
   5 nodes: planning, orchestrator, developer, tester, devops
   Hub-and-spoke: all agents route back to orchestrator
   Conditional edge: orchestrator decides next agent dynamically
   Self-healing loop: dev → test → fail → dev → test → pass → devops


In [259]:
# ============================================================================
# STEP 8.2: run_pipeline() — Execute the Full Pipeline
# ============================================================================
#
# WHAT THIS IS:
# -------------
# This is the TOP-LEVEL ENTRY POINT for running the entire multi-agent
# system. The Streamlit UI calls this when the user clicks "Run Pipeline".
#
# WHAT IT DOES:
# 1. Calls build_pipeline() to construct the LangGraph StateGraph
# 2. Creates the initial state with all required fields
# 3. Invokes the graph — this BLOCKS until the orchestrator routes to END
# 4. Returns the final state with ALL generated artefacts
#
# WHAT HAPPENS INSIDE graph.invoke():
# ────────────────────────────────────
#   Planning node → sets current_phase = "planning_complete"
#   Orchestrator  → reads phase → decides "developer"
#   Developer     → generates code → sets phase = "development_complete"
#   Orchestrator  → reads phase → decides "tester"
#   Tester        → generates + runs tests
#     IF PASS:    → sets test_passed = True, phase = "testing_complete"
#                 → Orchestrator → decides "devops"
#     IF FAIL:    → sets test_passed = False, retry_count += 1
#                 → Orchestrator → decides "developer" (fix loop!)
#                 → Developer fixes code → Tester re-runs (up to 3x)
#   DevOps        → generates configs, saves files, creates PR
#                 → sets phase = "devops_complete"
#   Orchestrator  → reads phase → decides "end"
#   → DONE!
#
# TYPICAL CALL COUNT (happy path, tests pass first try):
#   Planning(1) + Orchestrator(5) + Developer(1) + Tester(1) + DevOps(1) = 9 nodes
#   Each node with an LLM call = ~$0.10-0.30 total
#   Time: 2-5 minutes


def run_pipeline(spec: dict, guidelines_context: str, project_name: str, db_schema: str = "") -> dict:
    """
    Execute the full multi-agent build pipeline.
    
    Parameters:
      spec              — Confirmed spec dict from BA Agent
      guidelines_context — Legacy param (each agent now uses its own RAG retriever)
      project_name      — Name of the project (used for git branch, file paths)
      db_schema         — Optional DB schema string (passed to developer)
    
    Returns:
      Final AgentState dict containing ALL generated artefacts:
        - planning_output  → {jira_tickets, confluence_doc, flowchart, readme}
        - api_code         → {filename: source_code}
        - test_code        → {filename: test_source_code}
        - test_results     → pytest output string
        - test_passed      → True/False
        - devops_config    → {filename: config_content}
        - pr_result        → {success, pr_url, pr_number}
        - build_log        → chronological list of all agent actions
    """
    # Step 1: Build the compiled LangGraph pipeline
    graph = build_pipeline()

    print("\n" + "=" * 60)
    print("🚀 PIPELINE STARTED")
    print(f"   Project: {project_name}")
    print(f"   RAG mode: each agent retrieves relevant guidelines")
    print(f"   DB schema: {'provided' if db_schema else 'not provided'}")
    print("=" * 60)

    # Step 2: Construct the initial state — every field must be initialized
    # LangGraph will complain if a node tries to read a missing field
    initial_state = {
        "messages": [],                # LangGraph message accumulator (starts empty)
        "spec": spec,                  # The confirmed API spec (drives everything)
        "guidelines_context": guidelines_context,  # Legacy — agents use own RAG now
        "db_schema": db_schema,        # For developer to know real table structure
        "project_name": project_name,  # Used for git branch name, output directory
        "planning_output": {},         # Will be filled by planning_node
        "api_code": {},                # Will be filled by development_node
        "test_code": {},               # Will be filled by testing_node
        "test_results": "",            # Will be filled by testing_node
        "test_passed": False,          # Will be set by testing_node
        "devops_config": {},           # Will be filled by devops_node
        "pr_result": {},               # Will be filled by devops_node
        "current_phase": "started",    # Orchestrator tracks this
        "next_agent": "",              # Orchestrator writes this
        "retry_count": 0,              # Tracks dev↔test fix loop iterations
        "max_retries": 3,              # Developer gets at most 3 fix attempts
        "build_log": ["[Pipeline] Multi-agent build pipeline started."],
    }

    # Thread ID for MemorySaver checkpointing (allows resuming from any node)
    config = {"configurable": {"thread_id": project_name}}

    # Step 3: Invoke the graph — this BLOCKS until orchestrator says "end"
    # The graph executes nodes in sequence, following edges and conditional routes
    final_state = graph.invoke(initial_state, config)

    # Step 4: Print the build log (chronological record of everything)
    print("\n" + "=" * 60)
    print("🏁 PIPELINE COMPLETE")
    for entry in final_state.get("build_log", []):
        print(f"   {entry}")
    print("=" * 60 + "\n")

    return final_state


print("✅ run_pipeline() defined!")
print("   Call with: final_state = run_pipeline(spec, '', 'my-project', db_schema)")
print("   Blocks until all agents finish. Returns final state with all artefacts.")

✅ run_pipeline() defined!
   Call with: final_state = run_pipeline(spec, '', 'my-project', db_schema)
   Blocks until all agents finish. Returns final state with all artefacts.


---
## 📌 Section 9: Demo — Run the System End-to-End

Now that all components are defined, let's run the system!

**Step 1**: Use the BA Agent to have a conversation and confirm a spec.  
**Step 2**: Run the full pipeline with the confirmed spec.

> ⚠️ **Requires** a valid `OPENAI_API_KEY` in your `.env` file.  
> The pipeline will call GPT-4o multiple times (BA, Planning, Developer, Tester, DevOps).

In [260]:
# ============================================================================
# STEP 9.1: Demo — BA Agent Conversation
# ============================================================================
#
# WHAT THIS DEMONSTRATES:
# -----------------------
# This is how the BA Agent works in practice. In the Streamlit app, this
# is an interactive chat — here we simulate a single-turn interaction.
#
# WHAT HAPPENS WHEN YOU RUN THIS CELL:
# 1. Create a BAAgent instance (its own ChatOpenAI with temperature=0.3)
# 2. Define a user request (e.g., "I want a REST API for a library system")
# 3. Use RAG to retrieve relevant BA guidelines from ba_guidelines.txt
# 4. Get the DB schema so the BA knows what data exists
# 5. Call conduct_conversation() with an empty chat history (first message)
# 6. The BA responds with:
#    - What APIs can be built with existing data
#    - Clarifying questions (3-4 at a time)
#    - Standards it will apply (from guidelines)
#
# IN THE STREAMLIT UI:
# - This would be a multi-turn chat
# - User answers the BA's questions
# - After enough detail, BA says "[SPEC_CONFIRMED]"
# - Then extract_spec() is called to produce the JSON spec
# - That JSON spec becomes the input to run_pipeline()

# Step 1: Create the BA Agent
ba_agent = BAAgent(
    model=settings.OPENAI_MODEL,
    api_key=settings.OPENAI_API_KEY,
)

# Step 2: Define what you want to build
user_request = "I want a REST API for a library management system with books and members"

# Step 3: Retrieve BA-specific guidelines via RAG
# This gets chunks from ba_guidelines.txt about API spec standards
retriever = get_retriever()
ba_guidelines = retriever.retrieve("API specification requirements gathering", top_k=5)

# Step 4: Get DB schema so the BA can reference existing tables
db_schema = db.get_schema_summary()

# Step 5: Call the BA Agent's conversation method
# chat_history=[] means this is the FIRST message in the conversation
ba_response = ba_agent.conduct_conversation(
    user_message=user_request,
    chat_history=[],           # Empty — first message
    guidelines_context=ba_guidelines,
    db_schema=db_schema,
)

# Step 6: Display the response
print("=" * 60)
print("📝 BA Agent Response:")
print("=" * 60)
print(ba_response)
print("\n💡 In the Streamlit UI, you would continue the conversation")
print("   until the BA confirms the spec (returns a JSON object).")


🧑‍💼 [BA Agent] Processing user message (72 chars)
✅ [BA Agent] Response generated (1095 chars)
📝 BA Agent Response:
To create a REST API for a library management system, we can start by defining the core entities and their relationships. Based on your request, we will focus on two primary resources: books and members. Let's explore the available data and how we can structure the API.

### Core Entities
1. **Books**: This will represent the library's collection of books.
2. **Members**: This will represent the individuals who are registered to borrow books.

### Initial Questions
1. **Books**: What specific details do you want to manage for each book (e.g., title, author, ISBN, availability)?
2. **Members**: What specific details do you want to manage for each member (e.g., name, membership ID, contact information)?
3. **Operations**: What operations should the API support (e.g., adding new books, registering members, borrowing books, returning books)?
4. **Authentication**: Do you req

In [261]:
# ============================================================================
# STEP 9.2: Demo — Run the Full Pipeline (with a sample spec)
# ============================================================================
#
# WHAT THIS DEMONSTRATES:
# -----------------------
# Instead of doing the full BA conversation, you can pass a pre-built
# spec directly to run_pipeline() and watch the entire system work:
#
#   sample_spec → run_pipeline()
#       → Planning (Scrum Master generates Jira/Confluence/README)
#       → Orchestrator → Developer (generates FastAPI code)
#       → Orchestrator → Tester (generates + runs pytest)
#       → (if fail) Orchestrator → Developer (fix) → Tester (retry)
#       → Orchestrator → DevOps (Docker/K8s/CI-CD + PR)
#       → END
#
# ⚠️ COST & TIME:
#   - Calls GPT-4o 7-15 times depending on fix loops
#   - Takes ~2-5 minutes
#   - Costs ~$0.10-0.30 in API credits
#
# WHAT YOU GET IN output_api/:
#   output_api/
#   ├── api/           ← FastAPI source code (main.py, models, routes, etc.)
#   ├── tests/         ← Self-contained pytest test files
#   ├── devops/        ← Dockerfile, docker-compose, K8s manifests, CI/CD
#   └── planning/      ← jira_tickets.md, confluence_doc.md, flowchart.md, readme.md

# --- Sample confirmed spec (simulates what the BA Agent would produce) ---
sample_spec = {
    "project_name": "library-api",
    "description": "REST API for a library management system",
    "entities": [
        {
            "name": "Book",
            "fields": {
                "id": "integer, primary key, auto-increment",
                "title": "string, required",
                "author": "string, required",
                "isbn": "string, unique",
                "available": "boolean, default true",
            },
        },
        {
            "name": "Member",
            "fields": {
                "id": "integer, primary key, auto-increment",
                "name": "string, required",
                "email": "string, unique, required",
                "joined_date": "date, auto-set",
            },
        },
    ],
    "endpoints": [
        "GET /books - list all books",
        "POST /books - add a new book",
        "GET /books/{id} - get book by id",
        "PUT /books/{id} - update a book",
        "DELETE /books/{id} - delete a book",
        "GET /members - list all members",
        "POST /members - register a new member",
    ],
    "tech_stack": "FastAPI + SQLAlchemy + SQLite",
    "auth": "None (public API)",
}

# Get DB schema so the developer knows the real table structure
db_schema = db.get_schema_summary()

# --- Run the full pipeline ---
# Uncomment the next 2 lines to actually execute:
# final_state = run_pipeline(sample_spec, "", "library-api", db_schema)
# print("Done! Check the output_api/ directory for generated files.")

print("📋 Sample spec defined. Uncomment the run_pipeline() call above to execute.")
print(f"   Project: {sample_spec['project_name']}")
print(f"   Entities: {[e['name'] for e in sample_spec['entities']]}")
print(f"   Endpoints: {len(sample_spec['endpoints'])}")
print(f"\n   When run, the pipeline will:")
print(f"   1. Generate Jira tickets, Confluence doc, flowchart, README")
print(f"   2. Generate FastAPI code for {len(sample_spec['entities'])} entities")
print(f"   3. Generate + execute self-contained pytest tests")
print(f"   4. Fix code if tests fail (up to 3 retries)")
print(f"   5. Generate Docker, K8s, CI/CD configs")
print(f"   6. Create a GitHub PR (if credentials configured)")

📋 Sample spec defined. Uncomment the run_pipeline() call above to execute.
   Project: library-api
   Entities: ['Book', 'Member']
   Endpoints: 7

   When run, the pipeline will:
   1. Generate Jira tickets, Confluence doc, flowchart, README
   2. Generate FastAPI code for 2 entities
   3. Generate + execute self-contained pytest tests
   4. Fix code if tests fail (up to 3 retries)
   5. Generate Docker, K8s, CI/CD configs
   6. Create a GitHub PR (if credentials configured)


---
## 📌 Section 10: Architecture Summary

### What We Built

| Component | File(s) | Purpose |
|-----------|---------|---------|
| **Configuration** | `config/settings.py` | Loads `.env`, centralises all config |
| **Database** | `database/db_manager.py` | SQLite with products, customers, orders |
| **RAG System** | `rag/loader.py`, `rag/retriever.py` | Per-agent FAISS vector stores + CrossEncoder reranker |
| **Utilities** | `utils/parsers.py`, `file_utils.py`, `git_utils.py` | JSON parsing, file I/O, git+PR operations |
| **Prompts** | `agents/prompts.py` | 8 prompt templates (350 lines total) |
| **BA Agent** | `agents/ba_agent.py` | Pre-pipeline: multi-turn spec conversation |
| **Scrum Master** | `agents/scrum_master.py` | Planning + Orchestration (hub of hub-and-spoke) |
| **Developer** | `agents/developer.py` | Code generation + self-healing code fixes |
| **Tester** | `agents/tester.py` | Test generation + pytest execution |
| **DevOps** | `agents/devops.py` | Docker/K8s/CI-CD + GitHub PR |
| **Pipeline** | `agents/graph.py` | LangGraph StateGraph wiring |
| **UI** | `app.py` | Streamlit with 6 tabs |

### Pipeline Flow (what happens when you click "Run Pipeline")

```
User → BA Agent (chat) → Confirmed Spec
                              ↓
                    Planning (Scrum Master)
                              ↓
                    Orchestrator → Developer → Orchestrator → Tester
                              ↓                                  ↓
                          (if pass)                        (if fail)
                              ↓                                  ↓
                           DevOps                          Developer (fix)
                              ↓                                  ↓
                            END                            Tester (retry)
                                                           (up to 3x)
```

### Key Design Decisions

1. **5 separate agents** (not one monolithic LLM call) — each with own prompt, LLM, and RAG retriever
2. **Hub-and-spoke** — orchestrator is the brain; all agents report back to it
3. **Self-healing loop** — tester fails → developer fixes → tester retries (max 3x)
4. **Per-agent RAG** — each agent gets only the guidelines relevant to its role
5. **Spec-driven** — everything flows from the confirmed spec JSON